<a href="https://colab.research.google.com/github/Abhay-287/Text-to-ISL-comparative-study/blob/main/text2gloss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision torchaudio transformers nltk scikit-learn tabulate rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=80acf2ad5328a562d988dced82e8e59e7375fb7fe0d2fd3831647a2d86cf3cf5
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [2]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import sentence_bleu,corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer

In [3]:
# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [5]:
raw_dataset = [
    ("The girl doesn't cook a fish.", 'GIRL FISH COOK NOT'),
    ('The teacher is looking at the house.', 'TEACHER HOUSE LOOK'),
    ('The girl is eating an apple.', 'GIRL APPLE EAT'),
    ('They are eating an apple.', 'THEY APPLE EAT'),
    ('They are writing a letter.', 'THEY LETTER WRITE'),
    ('He is drinking a tea.', 'HE TEA DRINK'),
    ('He will buy dress next week.', 'NEXT WEEK HE DRESS BUY'),
    ('He is cooking a food.', 'HE FOOD COOK'),
    ('The boy is cleaning floor.', 'BOY FLOOR CLEAN'),
    ('They are watching a bird.', 'THEY BIRD WATCH'),
    ('They read letter this morning.', 'MORNING THEY LETTER READ'),
    ('She is eating an apple now.', 'NOW SHE APPLE EAT'),
    ('The doctor is eating a mango.', 'DOCTOR MANGO EAT'),
    ('Why did my mother take a car?', 'MY MOTHER CAR TAKE WHY'),
    ('He cooked food yesterday.', 'YESTERDAY HE FOOD COOK FINISH'),
    ('My mother is liking a mango.', 'MY MOTHER MANGO LIKE'),
    ('She is buying a book now.', 'NOW SHE BOOK BUY'),
    ('The girl is giving food.', 'GIRL FOOD GIVE'),
    ('My father is drinking milk.', 'MY FATHER MILK DRINK'),
    ('He will drink water tomorrow.', 'TOMORROW HE WATER DRINK WILL'),
    ('What do they want?', 'THEY WANT WHAT'),
    ('The doctor is buying a car.', 'DOCTOR CAR BUY'),
    ("They don't cook food.", 'THEY FOOD COOK NOT'),
    ('My father is liking a book.', 'MY FATHER BOOK LIKE'),
    ("The boy doesn't buy a shirt.", 'BOY SHIRT BUY NOT'),
    ('Why did my mother take clothes?', 'MY MOTHER CLOTHES TAKE WHY'),
    ('He is buying a house.', 'HE HOUSE BUY'),
    ('The teacher is writing with pen.', 'TEACHER PEN WRITE'),
    ('The doctor is cooking food.', 'DOCTOR FOOD COOK'),
    ('She is drinking an milk now.', 'NOW SHE MILK DRINK'),
    ('Where is the girl going with clothes?', 'GIRL CLOTHES GO WHERE'),
    ('We are making a sketch.', 'WE SKETCH MAKE'),
    ('The girl is driving a car.', 'GIRL CAR DRIVE'),
    ('I am buying clothes.', 'I CLOTHES BUY'),
    ('My mother is drinking water.', 'MY MOTHER WATER DRINK'),
    ('My father is going to office.', 'MY FATHER OFFICE GO'),
    ('Where is he going with a gun?', 'HE GUN GO WHERE'),
    ('The teacher is collecting money.', 'TEACHER MONEY COLLECT'),
    ('My mother is washing a clothes.', 'MY MOTHER CLOTHES WASH'),
    ('My father is reading a book.', 'MY FATHER BOOK READ'),
    ('The teacher is giving money.', 'TEACHER MONEY GIVE'),
    ('My father is cooking food.', 'MY FATHER FOOD COOK'),
    ('He is reading a book.', 'HE BOOK READ'),
    ('She read book last week.', 'PAST WEEK SHE BOOK READ FINISH'),
    ('The doctor is giving a medicine.', 'DOCTOR MEDICINE GIVE'),
    ('I am having a dinner.', 'I DINNER EAT'),
    ('The teacher is checking a notebook.', 'TEACHER NOTEBOOK CHECK'),
    ('Where is your father going with car?', 'YOUR FATHER car GO WHERE'),
    ('My friend is taking drugs.', 'MY FRIEND DRUGS TAKE'),
    ('Why is my father going to market?', 'MY FATHER MARKET GO WHY'),
    ('I am drinking a milk tonight.', 'NIGHT I MILK DRINK'),
    ('They are drinking alcohol tonight.', 'NIGHT THEY ALCOHOL DRINK'),
    ("They don't buy a house.", 'THEY HOUSE BUY NOT'),
    ('She is reading a novel.', 'SHE NOVEL READ'),
    ('We are making a picture.', 'WE PICTURE MAKE'),
    ('He is playing with ball.', 'HE BALL PLAY'),
    ("I don't drink a wine.", 'I WINE DRINK NOT'),
    ('The boy is hunting a bird.', 'BOY BIRD HUNT'),
    ('We ate food yesterday.', 'YESTERDAY WE FOOD EAT FINISH'),
    ('He is cooking food tonight.', 'NIGHT HE WATER FOOD'),
    ('He is reading book now.', 'NOW HE BOOK READ'),
    ("He doesn't book a phone.", 'HE PHONE BOOK NOT'),
    ('They want a computer.', 'THEY COMPUTER WANT'),
    ("The doctor doesn't ate food.", 'DOCTOR FOOD EAT NOT'),
    ('The girl is drinking water.', 'GIRL WATER DRINK'),
    ('Where is she going with clothes?', 'SHE CLOTHES GO WHERE'),
    ('The teacher is going to school.', 'TEACHER SCHOOL GO'),
    ('They are buying a phone now.', 'NOW THEY PHONE BUY'),
    ('The doctor is giving a medicine.', 'DOCTOR MEDICINE GIVE'),
    ('The teacher is taking exam.', 'TEACHER EXAM TAKE'),
    ('Why did your brother take a computer?', 'YOUR BROTHER COMPUTER TAKE WHY'),
    ('I live in kolkata.', 'I KOLKATA LIVE'),
    ('They are cooking food.', 'THEY FOOD COOK'),
    ('They drank beer last week.', 'PAST WEEK THEY BEER DRINK FINISH'),
    ('I am wearing clothes.', 'I CLOTHES WEAR'),
    ('My mother is washing a utensils.', 'MY MOTHER UTENSILS WASH'),
    ("I don't read a book.", 'I BOOK READ NOT'),
    ('My friend is washing a car.', 'MY FRIEND CAR WASH'),
    ('We are cleaning floor.', 'WE FLOOR CLEAN'),
    ("She doesn't read a book.", 'SHE BOOK READ NOT'),
    ('The girl is liking a computer.', 'GIRL COMPUTER LIKE'),
    ('They are wanting a house.', 'THEY HOUSE WANT'),
    ("The boy doesn't use a phone.", 'BOY PHONE USE NOT'),
    ('My father is buying a bird.', 'MY FATHER BIRD BUY'),
    ('We are purchasing a house.', 'WE HOUSE PURCHASE'),
    ('The teacher is writing a letter.', 'TEACHER LETTER WRITE'),
    ('I am eating a mango today.', 'TODAY I MANGO EAT'),
    ('He is washing clothes.', 'HE CLOTHES WASH'),
    ('They read novel yesterday.', 'YESTERDAY THEY NOVEL READ FINISH'),
    ('They ate a mango last week.', 'PAST WEEK THEY MANGO EAT FINISH'),
    ("The boy doesn't like bird.", 'BOY BIRD LIKE NOT'),
    ('Where are they going with a letter?', 'THEY LETTER GO WHERE'),
    ('I ate a mango this morning.', 'MORNING I MANGO EAT FINISH'),
    ('Why did the doctor insult patient?', 'DOCTOR PATIENT INSULT WHY'),
    ('The doctor is eating food.', 'DOCTOR FOOD EAT'),
    ('The teacher is talking from a phone.', 'TEACHER PHONE TALK'),
    ('They are counting money.', 'THEY MONEY COUNT'),
    ('My mother is buying a book.', 'MY MOTHER BOOK BUY'),
    ('Why are they buying a car ?', 'THEY CAR BUY WHY'),
    ("He doesn't read a book.", 'HE BOOK READ NOT'),
    ("I don't want money.", 'I MONEY WANT NOT'),
    ('The boy is going school.', 'BOY SCHOOL GO'),
    ("The girl doesn't read a book.", 'GIRL BOOK READ NOT'),
    ("The teacher doesn't go to school.", 'TEACHER SCHOOL GO NOT'),
    ('My mother is baking a cake.', 'MY MOTHER CAKE BAKE'),
    ('My friend is eating a mango.', 'MY FRIEND MANGO EAT'),
    ('Why did she take a letter?', 'SHE LETTER TAKE WHY'),
    ('He is giving money.', 'HE MONEY GIVE'),
    ('We are liking food.', 'WE FOOD LIKE'),
    ("My friend doesn't buy water.", 'MY FRIEND WATER BUY NOT'),
    ('They are taking a computer.', 'THEY COMPUTER TAKE'),
    ("They don't take money.", 'THEY MONEY TAKE NOT'),
    ("He doesn't need a computer.", 'HE COMPUTER NEED NOT'),
    ('My father is wanting a house.', 'MY FATHER HOUSE WANT'),
    ('My mother is taking water.', 'MY MOTHER WATER TAKE'),
    ('My friend is giving food.', 'MY FRIEND FOOD GIVE'),
    ('My friend is drinking milk.', 'MY FRIEND MILK DRINK'),
    ('I am reading a book.', 'I BOOK READ'),
    ('My friend is drinking milk.', 'MY FRIEND MILK DRINK'),
    ('He is wanting a mango.', 'HE MANGO WANT'),
    ('Why did the boy take money?', 'BOY MONEY TAKE WHY'),
    ('I am drinking juice.', 'I JUICE DRINK'),
    ('The boy is washing clothes.', 'BOY CLOTHES WASH'),
    ('My father is changing clothes.', 'MY FATHER CLOTHES CHANGE'),
    ('My mother is counting money.', 'MY MOTHER MONEY COUNT'),
    ('The doctor is drinking juice.', 'DOCTOR JUICE DRINK'),
    ('He is taking a book.', 'HE BOOK TAKE'),
    ('The boy is buying notebook.', 'BOY NOTEBOOK BUY'),
    ("My father doesn't cook food.", 'MY FATHER FOOD COOK NOT'),
    ("My friend doesn't need money.", 'MY FRIEND MONEY NEED NOT'),
    ("She doesn't drink wine.", 'SHE WINE DRINK NOT'),
    ('They are writing a letter.', 'THEY LETTER WRITE'),
    ('The girl is buying water.', 'GIRL WATER BUY'),
    ('We are building a house.', 'WE HOUSE BUILD'),
    ('The doctor is buying food.', 'DOCTOR FOOD BUY'),
    ('We ate food yesterday.', 'YESTERDAY WE FOOD EAT FINISH'),
    ('My brother is wasting money.', 'MY BROTHER MONEY WASTE'),
    ('We are watching movie.', 'WE MOVIE WATCH'),
    ('My father is cooking food.', 'MY FATHER FOOD COOK'),
    ('The boy is wanting a letter.', 'BOY LETTER WANT'),
    ('My father is seeing a letter.', 'MY FATHER LETTER SEE'),
    ('I read a book this morning.', 'MORNING I BOOK READ FINISH'),
    ("The boy doesn't drive a car.", 'BOY CAR DRIVE NOT'),
    ('My father is repairing a computer.', 'MY FATHER COMPUTER REPAIR'),
    ('He cooked an egg last week.', 'PAST WEEK HE EGG COOK FINISH'),
    ('I am taking money.', 'I MONEY TAKE'),
    ('Why did my father take a book?', 'MY FATHER BOOK TAKE WHY'),
    ('The girl is driving a car.', 'GIRL CAR DRIVE'),
    ('The girl is drinking juice.', 'GIRL JUICE DRINK'),
    ("I don't use a phone.", 'I PHONE USE NOT'),
    ('My mother is cooking food.', 'MY MOTHER FOOD COOK'),
    ('He is buying clothes.', 'HE CLOTHES BUY'),
    ('He will go to dubai next week.', 'NEXT WEEK HE DUBAI GO WILL'),
    ("My father doesn't buy water.", 'MY FATHER WATER BUY NOT'),
    ('He is taking a phone.', 'HE PHONE TAKE'),
    ('My friend is giving a computer.', 'MY FRIEND COMPUTER GIVE'),
    ('I bought an apple yesterday.', 'YESTERDAY I APPLE BUY FINISH'),
    ('The teacher is drinking water.', 'TEACHER WATER DRINK'),
    ('My father is buying a phone.', 'MY FATHER PHONE BUY'),
    ('My mother is wanting an apple.', 'MY MOTHER APPLE WANT'),
    ('Why did the boy take an apple?', 'BOY APPLE TAKE WHY'),
    ("The boy doesn't buy a computer.", 'BOY COMPUTER BUY NOT'),
    ('Where is my mother going with an apple?', 'MY MOTHER APPLE GO WHERE'),
    ('We will go to delhi tomorrow.', 'TOMORROW WE DELHI GO WILL'),
    ('Where is my friend going with food?', 'MY FRIEND FOOD GO WHERE'),
    ('My mother is watching clouds.', 'MY MOTHER CLOUDS WATCH'),
    ('The mechanic is washing a car.', 'MECHANIC CAR WASH'),
    ('They will read letter tomorrow.', 'TOMORROW THEY LETTER READ WILL'),
    ('She is cleaning a phone.', 'SHE PHONE CLEAN'),
    ('We ate an apple last week.', 'PAST WEEK WE APPLE EAT FINISH'),
    ("We don't eat food.", 'WE FOOD EAT NOT'),
    ('Why did my friend take an apple?', 'MY FRIEND APPLE TAKE WHY'),
    ('They will publish paper next week.', 'NEXT WEEK THEY PAPER PUBLISH WILL'),
    ('I am taking a phone.', 'I PHONE TAKE'),
    ('I am buying a mango tonight.', 'NIGHT I MANGO BUY'),
    ('I am hunting a bird.', 'I BIRD HUNT'),
    ('The officer is taking money.', 'OFFICER MONEY WRITE'),
    ('They bought food yesterday.', 'YESTERDAY THEY FOOD BUY FINISH'),
    ('They are eating pizza.', 'THEY PIZZA EAT'),
    ('We are buying a house now.', 'NOW WE HOUSE BUY'),
    ('She is giving water.', 'SHE WATER GIVE'),
    ('My friend is eating a mango.', 'MY FRIEND MANGO EAT'),
    ("I don't want burger.", 'I BURGER WANT NOT'),
    ('The doctor is examining a patient.', 'DOCTOR PATIENT EXAMINE'),
    ('He is buying an apple.', 'HE APPLE BUY'),
    ('I am taking a mango.', 'I MANGO TAKE'),
    ('The teacher is liking a book.', 'TEACHER BOOK LIKE'),
    ('The teacher is cooking meal.', 'TEACHER MEAL COOK'),
    ("The doctor doesn't read a book.", 'DOCTOR BOOK READ NOT'),
    ('We are buying food tonight.', 'NIGHT WE FOOD BUY'),
    ("My father doesn't read a book.", 'MY FATHER BOOK READ NOT'),
    ('I am giving water.', 'I WATER GIVE'),
    ("The teacher doesn't kill the bird.", 'TEACHER BIRD KILL NOT'),
    ('We are liking a book.', 'WE BOOK LIKE'),
    ("She doesn't buy milk.", 'SHE MILK BUY NOT'),
    ('I am liking therapy.', 'I THERAPY LIKE'),
    ('I am driving a ship.', 'I SHIP DRIVE'),
    ("My friend doesn't eat chiken.", 'MY FRIEND CHIKEN EAT NOT'),
    ('My mother is boiling milk.', 'MY MOTHER MILK BOIL'),
    ("My mother doesn't drive a car.", 'MY MOTHER CAR DRIVE NOT'),
    ('I is cooking a mango now.', 'NOW I MANGO COOK'),
    ('She is cooking a letter.', 'SHE LETTER COOK'),
    ('We are washing a phone.', 'WE PHONE WASH'),
    ('He is taking a computer.', 'HE COMPUTER TAKE'),
    ("My friend doesn't drink money.", 'MY FRIEND MONEY DRINK NOT'),
    ('The doctor is washing water.', 'DOCTOR WATER WASH'),
    ('I am making milk.', 'I MILK MAKE'),
    ('Where is the teacher going with an apple?', 'TEACHER APPLE GO WHERE'),
    ('He is drinking water tonight.', 'NIGHT HE WATER DRINK'),
    ('My friend is drinking a car.', 'MY FRIEND CAR DRINK'),
    ("My father doesn't read a letter.", 'MY FATHER LETTER READ NOT'),
    ('My mother is giving money.', 'MY MOTHER MONEY GIVE'),
    ("My friend doesn't cook milk.", 'MY FRIEND MILK COOK NOT'),
    ("They don't cook a book.", 'THEY BOOK COOK NOT'),
    ('She read a mango yesterday.', 'YESTERDAY SHE MANGO READ FINISH'),
    ('He is making school.', 'HE SCHOOL MAKE'),
    ('My mother is writing a mango.', 'MY MOTHER MANGO WRITE'),
    ("We don't eat money.", 'WE MONEY EAT NOT'),
    ('She is buying an apple now.', 'NOW SHE APPLE BUY'),
    ('She will cook a mango next week.', 'NEXT WEEK SHE MANGO COOK WILL'),
    ('She drank food yesterday.', 'YESTERDAY SHE FOOD DRINK FINISH'),
    ('The boy is taking a car.', 'BOY CAR TAKE'),
    ('She is taking a letter.', 'SHE LETTER TAKE'),
    ('I is buying an apple tonight.', 'NIGHT I APPLE BUY'),
    ('She cooked a book last week.', 'PAST WEEK SHE BOOK COOK FINISH'),
    ('We is cooking a mango now.', 'NOW WE MANGO COOK'),
    ('My mother is washing a phone.', 'MY MOTHER PHONE WASH'),
    ('The teacher is washing a bird.', 'TEACHER BIRD WASH'),
    ('He read water this morning.', 'MORNING HE WATER READ FINISH'),
    ('I am eating school.', 'I SCHOOL EAT'),
    ('The doctor is drinking a house.', 'DOCTOR HOUSE DRINK'),
    ('They will cook a book tomorrow.', 'TOMORROW THEY BOOK COOK WILL'),
    ('They are taking a house.', 'THEY HOUSE TAKE'),
    ('I drank an apple yesterday.', 'YESTERDAY I APPLE DRINK FINISH'),
    ('My friend is making a letter.', 'MY FRIEND LETTER MAKE'),
    ('My father is washing water.', 'MY FATHER WATER WASH'),
    ('My mother is giving a house.', 'MY MOTHER HOUSE GIVE'),
    ('We are writing school.', 'WE SCHOOL WRITE'),
    ('We are writing a book.', 'WE BOOK WRITE'),
    ('We ate an apple last week.', 'PAST WEEK WE APPLE EAT FINISH'),
    ("The teacher doesn't read money.", 'TEACHER MONEY READ NOT'),
    ('My friend is buying a computer.', 'MY FRIEND COMPUTER BUY'),
    ('My mother is driving school.', 'MY MOTHER SCHOOL DRIVE'),
    ("The doctor doesn't drink a phone.", 'DOCTOR PHONE DRINK NOT'),
    ('We are cooking water.', 'WE WATER COOK'),
    ('She is buying a book today.', 'TODAY SHE BOOK BUY'),
    ('He is driving a bird.', 'HE BIRD DRIVE'),
    ('He is eating an apple.', 'HE APPLE EAT'),
    ("We don't cook a house.", 'WE HOUSE COOK NOT'),
    ("They don't eat a letter.", 'THEY LETTER EAT NOT'),
    ('He is giving a computer.', 'HE COMPUTER GIVE'),
    ('We are seeing water.', 'WE WATER SEE'),
    ("The teacher doesn't cook a book.", 'TEACHER BOOK COOK NOT'),
    ("We don't cook a computer.", 'WE COMPUTER COOK NOT'),
    ('He is driving a computer.', 'HE COMPUTER DRIVE'),
    ('We are eating an apple.', 'WE APPLE EAT'),
    ('She is driving a book.', 'SHE BOOK DRIVE'),
    ("My friend doesn't buy a bird.", 'MY FRIEND BIRD BUY NOT'),
    ('The girl is writing an apple.', 'GIRL APPLE WRITE'),
    ("She doesn't drink water.", 'SHE WATER DRINK NOT'),
    ('She is eating a book tonight.', 'NIGHT SHE BOOK EAT'),
    ('She is eating a mango tonight.', 'NIGHT SHE MANGO EAT'),
    ('I am eating water.', 'I WATER EAT'),
    ('We are wanting a letter.', 'WE LETTER WANT'),
    ('My friend is eating milk.', 'MY FRIEND MILK EAT'),
    ('The girl is buying food.', 'GIRL FOOD BUY'),
    ('She is cooking a book today.', 'TODAY SHE BOOK COOK'),
    ('The teacher is giving a computer.', 'TEACHER COMPUTER GIVE'),
    ("The boy doesn't cook clothes.", 'BOY CLOTHES COOK NOT'),
    ('He drank food this morning.', 'MORNING HE FOOD DRINK FINISH'),
    ('We are reading money.', 'WE MONEY READ'),
    ('I cooked food last week.', 'PAST WEEK I FOOD COOK FINISH'),
    ('She is eating school.', 'SHE SCHOOL EAT'),
    ('Where is she going with a mango?', 'SHE MANGO GO WHERE'),
    ('The boy is giving a house.', 'BOY HOUSE GIVE'),
    ('My father is cooking a bird.', 'MY FATHER BIRD COOK'),
    ('I is cooking an apple now.', 'NOW I APPLE COOK'),
    ('I am drinking food.', 'I FOOD DRINK'),
    ('They are driving clothes.', 'THEY CLOTHES DRIVE'),
    ('They are giving a bird.', 'THEY BIRD GIVE'),
    ('They ate a book yesterday.', 'YESTERDAY THEY BOOK EAT FINISH'),
    ('The boy is cooking water.', 'BOY WATER COOK'),
    ('He bought water yesterday.', 'YESTERDAY HE WATER BUY FINISH'),
    ('The doctor is writing food.', 'DOCTOR FOOD WRITE'),
    ('The doctor is washing clothes.', 'DOCTOR CLOTHES WASH'),
    ("They don't cook a letter.", 'THEY LETTER COOK NOT'),
    ('He will drink water tomorrow.', 'TOMORROW HE WATER DRINK WILL'),
    ('They is reading a mango today.', 'TODAY THEY MANGO READ'),
    ('The girl is wanting clothes.', 'GIRL CLOTHES WANT'),
    ('They will read food next week.', 'NEXT WEEK THEY FOOD READ WILL'),
    ('I am buying school.', 'I SCHOOL BUY'),
    ('We are taking water.', 'WE WATER TAKE'),
    ('The doctor is drinking a computer.', 'DOCTOR COMPUTER DRINK'),
    ('My father is drinking a phone.', 'MY FATHER PHONE DRINK'),
    ('We are washing a car.', 'WE CAR WASH'),
    ('My friend is cooking a house.', 'MY FRIEND HOUSE COOK'),
    ('I ate food this morning.', 'MORNING I FOOD EAT FINISH'),
    ('Where is they going with a phone?', 'THEY PHONE GO WHERE'),
    ('They are driving a book.', 'THEY BOOK DRIVE'),
    ("He doesn't cook clothes.", 'HE CLOTHES COOK NOT'),
    ('We are cooking a car.', 'WE CAR COOK'),
    ('She is buying a phone.', 'SHE PHONE BUY'),
    ('He is cooking food tonight.', 'NIGHT HE FOOD COOK'),
    ('He will read an apple next week.', 'NEXT WEEK HE APPLE READ WILL'),
    ('My father is wanting a computer.', 'MY FATHER COMPUTER WANT'),
    ('My friend is cooking a computer.', 'MY FRIEND COMPUTER COOK'),
    ('Where is we going with money?', 'WE MONEY GO WHERE'),
    ('She will cook a book next week.', 'NEXT WEEK SHE BOOK COOK WILL'),
    ('He is cleaning a mango.', 'HE MANGO CLEAN'),
    ('I am taking food.', 'I FOOD TAKE'),
    ('We are seeing money.', 'WE MONEY SEE'),
    ('He is buying water tonight.', 'NIGHT HE WATER BUY'),
    ('She is eating a mango today.', 'TODAY SHE MANGO EAT'),
    ("The teacher doesn't cook a house.", 'TEACHER HOUSE COOK NOT'),
    ('My friend is giving school.', 'MY FRIEND SCHOOL GIVE'),
    ('She is cooking an apple tonight.', 'NIGHT SHE APPLE COOK'),
    ('They are taking money.', 'THEY MONEY TAKE'),
    ("He doesn't read food.", 'HE FOOD READ NOT'),
    ("I don't cook clothes.", 'I CLOTHES COOK NOT'),
    ("She doesn't buy a house.", 'SHE HOUSE BUY NOT'),
    ('My friend is writing a car.', 'MY FRIEND CAR WRITE'),
    ('We is eating food today.', 'TODAY WE FOOD EAT'),
    ('She is reading water.', 'SHE WATER READ'),
    ('She is drinking an apple today.', 'TODAY SHE APPLE DRINK'),
    ("My mother doesn't eat a house.", 'MY MOTHER HOUSE EAT NOT'),
    ("He doesn't drink food.", 'HE FOOD DRINK NOT'),
    ('The doctor is washing food.', 'DOCTOR FOOD WASH'),
    ('They are eating a house.', 'THEY HOUSE EAT'),
    ('The doctor is driving water.', 'DOCTOR WATER DRIVE'),
    ('I am taking a letter.', 'I LETTER TAKE'),
    ('She ate food last week.', 'PAST WEEK SHE FOOD EAT FINISH'),
    ('They drank a mango yesterday.', 'YESTERDAY THEY MANGO DRINK FINISH'),
    ('My father is drinking clothes.', 'MY FATHER CLOTHES DRINK'),
    ('We will read food tomorrow.', 'TOMORROW WE FOOD READ WILL'),
    ("The doctor doesn't cook an apple.", 'DOCTOR APPLE COOK NOT'),
    ('She drank a book last week.', 'PAST WEEK SHE BOOK DRINK FINISH'),
    ('The boy is writing milk.', 'BOY MILK WRITE'),
    ("The teacher doesn't eat a computer.", 'TEACHER COMPUTER EAT NOT'),
    ("They don't buy a phone.", 'THEY PHONE BUY NOT'),
    ('We are buying a letter.', 'WE LETTER BUY'),
    ('My mother is drinking a computer.', 'MY MOTHER COMPUTER DRINK'),
    ('She is wanting school.', 'SHE SCHOOL WANT'),
    ('Where is the teacher going with milk?', 'TEACHER MILK GO WHERE'),
    ("My friend doesn't drink an apple.", 'MY FRIEND APPLE DRINK NOT'),
    ('The teacher is reading a house.', 'TEACHER HOUSE READ'),
    ('I bought food last week.', 'PAST WEEK I FOOD BUY FINISH'),
    ('Why did the girl take a mango?', 'GIRL MANGO TAKE WHY'),
    ("They don't eat school.", 'THEY SCHOOL EAT NOT'),
    ('The boy is cleaning a bird.', 'BOY BIRD CLEAN'),
    ("The boy doesn't drink an apple.", 'BOY APPLE DRINK NOT'),
    ('My mother is making an apple.', 'MY MOTHER APPLE MAKE'),
    ('They cooked food last week.', 'PAST WEEK THEY FOOD COOK FINISH'),
    ('I am cooking milk.', 'I MILK COOK'),
    ("The boy doesn't drink a mango.", 'BOY MANGO DRINK NOT'),
    ('The boy is writing a house.', 'BOY HOUSE WRITE'),
    ('The boy is cooking a mango.', 'BOY MANGO COOK'),
    ('They are buying a house.', 'THEY HOUSE BUY'),
    ("My father doesn't drink a car.", 'MY FATHER CAR DRINK NOT'),
    ("The boy doesn't cook a letter.", 'BOY LETTER COOK NOT'),
    ("We don't buy food.", 'WE FOOD BUY NOT'),
    ('My mother is taking a mango.', 'MY MOTHER MANGO TAKE'),
    ('The boy is giving water.', 'BOY WATER GIVE'),
    ('She is cleaning a book.', 'SHE BOOK CLEAN'),
    ('Why did the boy take clothes?', 'BOY CLOTHES TAKE WHY'),
    ('He is reading a mango tonight.', 'NIGHT HE MANGO READ'),
    ("The girl doesn't buy a mango.", 'GIRL MANGO BUY NOT'),
    ("The boy doesn't buy milk.", 'BOY MILK BUY NOT'),
    ('The boy is drinking a car.', 'BOY CAR DRINK'),
    ('We are reading clothes.', 'WE CLOTHES READ'),
    ('Why did they take a house?', 'THEY HOUSE TAKE WHY'),
    ("The girl doesn't cook food.", 'GIRL FOOD COOK NOT'),
    ('My father is washing food.', 'MY FATHER FOOD WASH'),
    ('The teacher is driving a letter.', 'TEACHER LETTER DRIVE'),
    ('The boy is making school.', 'BOY SCHOOL MAKE'),
    ('We are writing clothes.', 'WE CLOTHES WRITE'),
    ('The doctor is giving a book.', 'DOCTOR BOOK GIVE'),
    ('She is buying food today.', 'TODAY SHE FOOD BUY'),
    ('He is buying a phone.', 'HE PHONE BUY'),
    ('The doctor is liking a book.', 'DOCTOR BOOK LIKE'),
    ('We are cleaning a letter.', 'WE LETTER CLEAN'),
    ('The doctor is wanting money.', 'DOCTOR MONEY WANT'),
    ('I is cooking water tonight.', 'NIGHT I WATER COOK'),
    ('They will drink water tomorrow.', 'TOMORROW THEY WATER DRINK WILL'),
    ("My friend doesn't buy clothes.", 'MY FRIEND CLOTHES BUY NOT'),
    ("I don't buy clothes.", 'I CLOTHES BUY NOT'),
    ('Where is my friend going with a house?', 'MY FRIEND HOUSE GO WHERE'),
    ('I bought water this morning.', 'MORNING I WATER BUY FINISH'),
    ('We are seeing clothes.', 'WE CLOTHES SEE'),
    ('They are buying a phone.', 'THEY PHONE BUY'),
    ('I is eating a book now.', 'NOW I BOOK EAT'),
    ("My friend doesn't buy a mango.", 'MY FRIEND MANGO BUY NOT'),
    ('They drank water this morning.', 'MORNING THEY WATER DRINK FINISH'),
    ('I am wanting milk.', 'I MILK WANT'),
    ('I am seeing a letter.', 'I LETTER SEE'),
    ('He is eating a book today.', 'TODAY HE BOOK EAT'),
    ('He will cook food next week.', 'NEXT WEEK HE FOOD COOK WILL'),
    ('She is buying a computer.', 'SHE COMPUTER BUY'),
    ("They don't cook school.", 'THEY SCHOOL COOK NOT'),
    ('I read an apple last week.', 'PAST WEEK I APPLE READ FINISH'),
    ('My mother is wanting a book.', 'MY MOTHER BOOK WANT'),
    ('They drank food this morning.', 'MORNING THEY FOOD DRINK FINISH'),
    ('They are giving a computer.', 'THEY COMPUTER GIVE'),
    ('He is making water.', 'HE WATER MAKE'),
    ('The doctor is drinking a bird.', 'DOCTOR BIRD DRINK'),
    ('I drank water yesterday.', 'YESTERDAY I WATER DRINK FINISH'),
    ('The boy is making a letter.', 'BOY LETTER MAKE'),
    ("He doesn't buy water.", 'HE WATER BUY NOT'),
    ('The boy is making a computer.', 'BOY COMPUTER MAKE'),
    ('I will buy a mango next week.', 'NEXT WEEK I MANGO BUY WILL'),
    ('Where is the teacher going with a house?', 'TEACHER HOUSE GO WHERE'),
    ('The boy is washing a car.', 'BOY CAR WASH'),
    ('My friend is drinking water.', 'MY FRIEND WATER DRINK'),
    ('He will eat a mango next week.', 'NEXT WEEK HE MANGO EAT WILL'),
    ('Why did they take a phone?', 'THEY PHONE TAKE WHY'),
    ('He read food last week.', 'PAST WEEK HE FOOD READ FINISH'),
    ('I bought an apple this morning.', 'MORNING I APPLE BUY FINISH'),
    ('I am driving food.', 'I FOOD DRIVE'),
    ("We don't drink a book.", 'WE BOOK DRINK NOT'),
    ('We is reading food tonight.', 'NIGHT WE FOOD READ'),
    ('My mother is taking a book.', 'MY MOTHER BOOK TAKE'),
    ('The teacher is wanting a mango.', 'TEACHER MANGO WANT'),
    ('My mother is reading a house.', 'MY MOTHER HOUSE READ'),
    ('They will eat water next week.', 'NEXT WEEK THEY WATER EAT WILL'),
    ('He is cooking a mango now.', 'NOW HE MANGO COOK'),
    ('He is writing a letter.', 'HE LETTER WRITE'),
    ("My mother doesn't eat water.", 'MY MOTHER WATER EAT NOT'),
    ('My mother is making a bird.', 'MY MOTHER BIRD MAKE'),
    ('She is drinking a phone.', 'SHE PHONE DRINK'),
    ('I am driving a letter.', 'I LETTER DRIVE'),
    ('They are taking school.', 'THEY SCHOOL TAKE'),
    ("We don't buy school.", 'WE SCHOOL BUY NOT'),
    ('My friend is driving a house.', 'MY FRIEND HOUSE DRIVE'),
    ('We are drinking school.', 'WE SCHOOL DRINK'),
    ('We are cleaning clothes.', 'WE CLOTHES CLEAN'),
    ('My friend is cooking a car.', 'MY FRIEND CAR COOK'),
    ('My friend is writing a bird.', 'MY FRIEND BIRD WRITE'),
    ("We don't cook a phone.", 'WE PHONE COOK NOT'),
    ("We don't read clothes.", 'WE CLOTHES READ NOT'),
    ('My mother is eating milk.', 'MY MOTHER MILK EAT'),
    ('My mother is drinking clothes.', 'MY MOTHER CLOTHES DRINK'),
    ('I am reading a computer.', 'I COMPUTER READ'),
    ('They drank an apple this morning.', 'MORNING THEY APPLE DRINK FINISH'),
    ('She will read water tomorrow.', 'TOMORROW SHE WATER READ WILL'),
    ('They is drinking water now.', 'NOW THEY WATER DRINK'),
    ('She is liking a phone.', 'SHE PHONE LIKE'),
    ('My mother is wanting a phone.', 'MY MOTHER PHONE WANT'),
    ("They don't cook milk.", 'THEY MILK COOK NOT'),
    ('The girl is wanting a computer.', 'GIRL COMPUTER WANT'),
    ('My friend is buying a book.', 'MY FRIEND BOOK BUY'),
    ('The girl is taking milk.', 'GIRL MILK TAKE'),
    ('My friend is drinking milk.', 'MY FRIEND MILK DRINK'),
    ('My friend is liking a letter.', 'MY FRIEND LETTER LIKE'),
    ("The doctor doesn't drink a book.", 'DOCTOR BOOK DRINK NOT'),
    ('My mother is driving a book.', 'MY MOTHER BOOK DRIVE'),
    ('The boy is driving a letter.', 'BOY LETTER DRIVE'),
    ('He is reading a house.', 'HE HOUSE READ'),
    ('We bought a book this morning.', 'MORNING WE BOOK BUY FINISH'),
    ('I am taking milk.', 'I MILK TAKE'),
    ('We are cleaning milk.', 'WE MILK CLEAN'),
    ('They read a book this morning.', 'MORNING THEY BOOK READ FINISH'),
    ('The teacher is driving a house.', 'TEACHER HOUSE DRIVE'),
    ('The teacher is eating a letter.', 'TEACHER LETTER EAT'),
    ('We are writing a computer.', 'WE COMPUTER WRITE'),
    ('My mother is making a mango.', 'MY MOTHER MANGO MAKE'),
    ("We don't cook a book.", 'WE BOOK COOK NOT'),
    ("The doctor doesn't drink milk.", 'DOCTOR MILK DRINK NOT'),
    ('Where is the doctor going with water?', 'DOCTOR WATER GO WHERE'),
    ('They are washing an apple.', 'THEY APPLE WASH'),
    ('We bought water yesterday.', 'YESTERDAY WE WATER BUY FINISH'),
    ("The boy doesn't cook money.", 'BOY MONEY COOK NOT'),
    ('The boy is writing an apple.', 'BOY APPLE WRITE'),
    ('I am taking water.', 'I WATER TAKE'),
    ('Why did we take a house?', 'WE HOUSE TAKE WHY'),
    ('I will drink food tomorrow.', 'TOMORROW I FOOD DRINK WILL'),
    ('My mother is writing a car.', 'MY MOTHER CAR WRITE'),
    ('We are seeing a house.', 'WE HOUSE SEE'),
    ('She will buy an apple tomorrow.', 'TOMORROW SHE APPLE BUY WILL'),
    ('I am making a mango.', 'I MANGO MAKE'),
    ('He ate an apple yesterday.', 'YESTERDAY HE APPLE EAT FINISH'),
    ('She drank water yesterday.', 'YESTERDAY SHE WATER DRINK FINISH'),
    ('They are eating a book.', 'THEY BOOK EAT'),
    ('I is drinking food now.', 'NOW I FOOD DRINK'),
    ('My father is buying milk.', 'MY FATHER MILK BUY'),
    ('I will drink an apple tomorrow.', 'TOMORROW I APPLE DRINK WILL'),
    ("She doesn't drink a book.", 'SHE BOOK DRINK NOT'),
    ('They are making a house.', 'THEY HOUSE MAKE'),
    ('I drank a book last week.', 'PAST WEEK I BOOK DRINK FINISH'),
    ('He will drink water next week.', 'NEXT WEEK HE WATER DRINK WILL'),
    ("She doesn't buy a letter.", 'SHE LETTER BUY NOT'),
    ('The girl is drinking a bird.', 'GIRL BIRD DRINK'),
    ('The boy is liking a car.', 'BOY CAR LIKE'),
    ('The boy is wanting a book.', 'BOY BOOK WANT'),
    ('The teacher is writing a house.', 'TEACHER HOUSE WRITE'),
    ('Where is he going with clothes?', 'HE CLOTHES GO WHERE'),
    ('The boy is taking food.', 'BOY FOOD TAKE'),
    ('The doctor is cooking milk.', 'DOCTOR MILK COOK'),
    ('I am reading a mango.', 'I MANGO READ'),
    ('He is reading water.', 'HE WATER READ'),
    ('They are cooking a mango.', 'THEY MANGO COOK'),
    ('My friend is seeing water.', 'MY FRIEND WATER SEE'),
    ('Why did i take a mango?', 'I MANGO TAKE WHY'),
    ("She doesn't drink an apple.", 'SHE APPLE DRINK NOT'),
    ("My father doesn't drink a house.", 'MY FATHER HOUSE DRINK NOT'),
    ('She is cooking water.', 'SHE WATER COOK'),
    ('He is giving a mango.', 'HE MANGO GIVE'),
    ("The boy doesn't buy an apple.", 'BOY APPLE BUY NOT'),
    ('He drank a mango yesterday.', 'YESTERDAY HE MANGO DRINK FINISH'),
    ('They will eat an apple next week.', 'NEXT WEEK THEY APPLE EAT WILL'),
    ('The doctor is cooking a bird.', 'DOCTOR BIRD COOK'),
    ('My friend is making an apple.', 'MY FRIEND APPLE MAKE'),
    ('They are cleaning a letter.', 'THEY LETTER CLEAN'),
    ('He is eating money.', 'HE MONEY EAT'),
    ('The boy is drinking a book.', 'BOY BOOK DRINK'),
    ("He doesn't eat money.", 'HE MONEY EAT NOT'),
    ('He is taking water.', 'HE WATER TAKE'),
    ("The teacher doesn't read a book.", 'TEACHER BOOK READ NOT'),
    ('She is giving money.', 'SHE MONEY GIVE'),
    ('He is liking a computer.', 'HE COMPUTER LIKE'),
    ("I don't buy school.", 'I SCHOOL BUY NOT'),
    ('They are liking food.', 'THEY FOOD LIKE'),
    ('The girl is cooking school.', 'GIRL SCHOOL COOK'),
    ('The girl is wanting milk.', 'GIRL MILK WANT'),
    ("My friend doesn't drink food.", 'MY FRIEND FOOD DRINK NOT'),
    ('My mother is wanting a computer.', 'MY MOTHER COMPUTER WANT'),
    ('The boy is taking a computer.', 'BOY COMPUTER TAKE'),
    ('They are making a computer.', 'THEY COMPUTER MAKE'),
    ('The boy is driving water.', 'BOY WATER DRIVE'),
    ('My mother is buying money.', 'MY MOTHER MONEY BUY'),
    ('My father is drinking school.', 'MY FATHER SCHOOL DRINK'),
    ("I don't eat money.", 'I MONEY EAT NOT'),
    ('My mother is washing an apple.', 'MY MOTHER APPLE WASH'),
    ('My friend is making a car.', 'MY FRIEND CAR MAKE'),
    ('My father is cooking a book.', 'MY FATHER BOOK COOK'),
    ('They will drink food tomorrow.', 'TOMORROW THEY FOOD DRINK WILL'),
    ('We drank water this morning.', 'MORNING WE WATER DRINK FINISH'),
    ('The teacher is cleaning school.', 'TEACHER SCHOOL CLEAN'),
    ('I read a mango yesterday.', 'YESTERDAY I MANGO READ FINISH'),
    ('Why did my mother take a letter?', 'MY MOTHER LETTER TAKE WHY'),
    ('We are taking clothes.', 'WE CLOTHES TAKE'),
    ('I am seeing water.', 'I WATER SEE'),
    ('She is eating an apple tonight.', 'NIGHT SHE APPLE EAT'),
    ('My father is wanting a car.', 'MY FATHER CAR WANT'),
    ('The boy is cooking a house.', 'BOY HOUSE COOK'),
    ('I will read food next week.', 'NEXT WEEK I FOOD READ WILL'),
    ('Why did my friend take a bird?', 'MY FRIEND BIRD TAKE WHY'),
    ('He will eat a book next week.', 'NEXT WEEK HE BOOK EAT WILL'),
    ('We is eating a book tonight.', 'NIGHT WE BOOK EAT'),
    ('I am cleaning a phone.', 'I PHONE CLEAN'),
    ('They are wanting a car.', 'THEY CAR WANT'),
    ('We will drink food next week.', 'NEXT WEEK WE FOOD DRINK WILL'),
    ("The doctor doesn't eat a bird.", 'DOCTOR BIRD EAT NOT'),
    ('They is buying an apple today.', 'TODAY THEY APPLE BUY'),
    ('We are driving a mango.', 'WE MANGO DRIVE'),
    ('My friend is reading a computer.', 'MY FRIEND COMPUTER READ'),
    ('She is reading food now.', 'NOW SHE FOOD READ'),
    ("My father doesn't read a book.", 'MY FATHER BOOK READ NOT'),
    ('We ate water yesterday.', 'YESTERDAY WE WATER EAT FINISH'),
    ('She is cooking a bird.', 'SHE BIRD COOK'),
    ('I is eating a book today.', 'TODAY I BOOK EAT'),
    ('He is making food.', 'HE FOOD MAKE'),
    ('I will cook an apple next week.', 'NEXT WEEK I APPLE COOK WILL'),
    ('I am writing a mango.', 'I MANGO WRITE'),
    ('They are seeing a phone.', 'THEY PHONE SEE'),
    ('We are taking money.', 'WE MONEY TAKE'),
    ('He is writing milk.', 'HE MILK WRITE'),
    ('I drank water last week.', 'PAST WEEK I WATER DRINK FINISH'),
    ('She is buying water tonight.', 'NIGHT SHE WATER BUY'),
    ("The doctor doesn't drink a mango.", 'DOCTOR MANGO DRINK NOT'),
    ('I am driving an apple.', 'I APPLE DRIVE'),
    ('I am washing a letter.', 'I LETTER WASH'),
    ('My father is writing a car.', 'MY FATHER CAR WRITE'),
    ('She is cleaning milk.', 'SHE MILK CLEAN'),
    ('Why did he take a house?', 'HE HOUSE TAKE WHY'),
    ('He is drinking an apple tonight.', 'NIGHT HE APPLE DRINK'),
    ('My mother is reading water.', 'MY MOTHER WATER READ'),
    ('The doctor is writing water.', 'DOCTOR WATER WRITE'),
    ("The boy doesn't cook a house.", 'BOY HOUSE COOK NOT'),
    ('They bought a book last week.', 'PAST WEEK THEY BOOK BUY FINISH'),
    ('They are driving a car.', 'THEY CAR DRIVE'),
    ('We are wanting food.', 'WE FOOD WANT'),
    ('Why did we take school?', 'WE SCHOOL TAKE WHY'),
    ('My friend is eating a computer.', 'MY FRIEND COMPUTER EAT'),
    ('He cooked a book last week.', 'PAST WEEK HE BOOK COOK FINISH'),
    ('The doctor is making clothes.', 'DOCTOR CLOTHES MAKE'),
    ('I am cleaning an apple.', 'I APPLE CLEAN'),
    ('They are writing a car.', 'THEY CAR WRITE'),
    ('My mother is wanting school.', 'MY MOTHER SCHOOL WANT'),
    ('We are drinking food.', 'WE FOOD DRINK'),
    ('The boy is cooking food.', 'BOY FOOD COOK'),
    ('We read a book last week.', 'PAST WEEK WE BOOK READ FINISH'),
    ("He doesn't drink a house.", 'HE HOUSE DRINK NOT'),
    ('The teacher is drinking money.', 'TEACHER MONEY DRINK'),
    ('They are eating a mango.', 'THEY MANGO EAT'),
    ('Where is he going with a phone?', 'HE PHONE GO WHERE'),
    ('The girl is reading a bird.', 'GIRL BIRD READ'),
    ("I don't cook a bird.", 'I BIRD COOK NOT'),
    ('The teacher is seeing clothes.', 'TEACHER CLOTHES SEE'),
    ('Where is my friend going with a phone?', 'MY FRIEND PHONE GO WHERE'),
    ('We are drinking a computer.', 'WE COMPUTER DRINK'),
    ("They don't drink milk.", 'THEY MILK DRINK NOT'),
    ('The girl is buying a house.', 'GIRL HOUSE BUY'),
    ('They is eating food tonight.', 'NIGHT THEY FOOD EAT'),
    ('They are cleaning clothes.', 'THEY CLOTHES CLEAN'),
    ('They are liking a house.', 'THEY HOUSE LIKE'),
    ("We don't eat a mango.", 'WE MANGO EAT NOT'),
    ('I am drinking a house.', 'I HOUSE DRINK'),
    ('My father is washing a bike.', 'MY FATHER PHONE BIKE'),
    ('My friend is washing an apple.', 'MY FRIEND APPLE WASH'),
    ("We don't use a phone.", 'WE PHONE USE NOT'),
    ('Why did i take food?', 'I FOOD TAKE WHY'),
    ('My friend is driving water.', 'MY FRIEND WATER DRIVE'),
    ('The girl is liking money.', 'GIRL MONEY LIKE'),
    ('He is reading a book now.', 'NOW HE BOOK READ'),
    ('My friend is cleaning a mango.', 'MY FRIEND MANGO CLEAN'),
    ("The boy doesn't buy water.", 'BOY WATER BUY NOT'),
    ('The boy is liking a mango.', 'BOY MANGO LIKE'),
    ("The doctor doesn't eat water.", 'DOCTOR WATER EAT NOT'),
    ("We don't read a mango.", 'WE MANGO READ NOT'),
    ('The teacher is washing food.', 'TEACHER FOOD WASH'),
    ('We are liking water.', 'WE WATER LIKE'),
    ('The teacher is drinking milk.', 'TEACHER MILK DRINK'),
    ('Where is he going with milk?', 'HE MILK GO WHERE'),
    ('Where is she going with a bird?', 'SHE BIRD GO WHERE'),
    ('They read an apple this morning.', 'MORNING THEY APPLE READ FINISH'),
    ('The teacher is cleaning a letter.', 'TEACHER LETTER CLEAN'),
    ('The boy is taking money.', 'BOY MONEY TAKE'),
    ('I am seeing food.', 'I FOOD SEE'),
    ('My father is liking a bird.', 'MY FATHER BIRD LIKE'),
    ('My father is driving money.', 'MY FATHER MONEY DRIVE'),
    ('They are drinking an apple.', 'THEY APPLE DRINK'),
    ('He bought a mango this morning.', 'MORNING HE MANGO BUY FINISH'),
    ('The girl is driving an apple.', 'GIRL APPLE DRIVE'),
    ('The boy is driving a phone.', 'BOY PHONE DRIVE'),
    ("My father doesn't eat a house.", 'MY FATHER HOUSE EAT NOT'),
    ("My friend doesn't drink milk.", 'MY FRIEND MILK DRINK NOT'),
    ('She is giving a car.', 'SHE CAR GIVE'),
    ('My mother is wanting money.', 'MY MOTHER MONEY WANT'),
    ('They will read a book next week.', 'NEXT WEEK THEY BOOK READ WILL'),
    ("The teacher doesn't drink a computer.", 'TEACHER COMPUTER DRINK NOT'),
    ('She is cleaning a mango.', 'SHE MANGO CLEAN'),
    ('I is buying a book today.', 'TODAY I BOOK BUY'),
    ('He will read food tomorrow.', 'TOMORROW HE FOOD READ WILL'),
    ('I am giving clothes.', 'I CLOTHES GIVE'),
    ('I is cooking an apple today.', 'TODAY I APPLE COOK'),
    ("My father doesn't buy a book.", 'MY FATHER BOOK BUY NOT'),
    ('The girl is washing a phone.', 'GIRL PHONE WASH'),
    ('I is eating water now.', 'NOW I WATER EAT'),
    ('My father is driving a letter.', 'MY FATHER LETTER DRIVE'),
    ('My friend is cooking a letter.', 'MY FRIEND LETTER COOK'),
    ('He will eat food tomorrow.', 'TOMORROW HE FOOD EAT WILL'),
    ('Why did the teacher take a letter?', 'TEACHER LETTER TAKE WHY'),
    ('The boy is reading an apple.', 'BOY APPLE READ'),
    ('The doctor is liking a mango.', 'DOCTOR MANGO LIKE'),
    ('She is buying a house.', 'SHE HOUSE BUY'),
    ("They don't drink a book.", 'THEY BOOK DRINK NOT'),
    ('My father is writing an apple.', 'MY FATHER APPLE WRITE'),
    ('The boy is buying a mango.', 'BOY MANGO BUY'),
    ('She is cooking an apple.', 'SHE APPLE COOK'),
    ('My friend is washing a car.', 'MY FRIEND CAR WASH'),
    ('My friend is wanting a computer.', 'MY FRIEND COMPUTER WANT'),
    ("The doctor doesn't cook a bird.", 'DOCTOR BIRD COOK NOT'),
    ('We is cooking a mango today.', 'TODAY WE MANGO COOK'),
    ('They are giving a house.', 'THEY HOUSE GIVE'),
    ('My friend is giving milk.', 'MY FRIEND MILK GIVE'),
    ('My mother is eating water.', 'MY MOTHER WATER EAT'),
    ('My father is giving food.', 'MY FATHER FOOD GIVE'),
    ('She will eat a mango tomorrow.', 'TOMORROW SHE MANGO EAT WILL'),
    ('The boy is giving a letter.', 'BOY LETTER GIVE'),
    ("My mother doesn't buy a house.", 'MY MOTHER HOUSE BUY NOT'),
    ('We are liking a letter.', 'WE LETTER LIKE'),
    ('They is buying a mango today.', 'TODAY THEY MANGO BUY'),
    ('Where is the girl going with a book?', 'GIRL BOOK GO WHERE'),
    ('I is reading a mango tonight.', 'NIGHT I MANGO READ'),
    ('We are making clothes.', 'WE CLOTHES MAKE'),
    ('We bought a mango this morning.', 'MORNING WE MANGO BUY FINISH'),
    ('The girl is making a phone.', 'GIRL PHONE MAKE'),
    ('The teacher is liking a car.', 'TEACHER CAR LIKE'),
    ("She doesn't drink a computer.", 'SHE COMPUTER DRINK NOT'),
    ("She doesn't drink a house.", 'SHE HOUSE DRINK NOT'),
    ('Where is the doctor going with an apple?', 'DOCTOR APPLE GO WHERE'),
    ('He is driving school.', 'HE SCHOOL DRIVE'),
    ('They is eating water tonight.', 'NIGHT THEY WATER EAT'),
    ('The teacher is cleaning a mango.', 'TEACHER MANGO CLEAN'),
    ('I cooked an apple last week.', 'PAST WEEK I APPLE COOK FINISH'),
    ('We are buying a car.', 'WE CAR BUY'),
    ('My mother is giving an apple.', 'MY MOTHER APPLE GIVE'),
    ('We are seeing a mango.', 'WE MANGO SEE'),
    ('We read food yesterday.', 'YESTERDAY WE FOOD READ FINISH'),
    ('She ate a mango last week.', 'PAST WEEK SHE MANGO EAT FINISH'),
    ('He is drinking a house.', 'HE HOUSE DRINK'),
    ('My father is driving an apple.', 'MY FATHER APPLE DRIVE'),
    ('My mother is wanting a mango.', 'MY MOTHER MANGO WANT'),
    ('He bought an apple last week.', 'PAST WEEK HE APPLE BUY FINISH'),
    ("He doesn't cook a car.", 'HE CAR COOK NOT'),
    ('My father is wanting a mango.', 'MY FATHER MANGO WANT'),
    ('We is eating food tonight.', 'NIGHT WE FOOD EAT'),
    ('Where is we going with food?', 'WE FOOD GO WHERE'),
    ('I am cooking a bird.', 'I BIRD COOK'),
    ("My friend doesn't drink clothes.", 'MY FRIEND CLOTHES DRINK NOT'),
    ('The teacher is drinking a letter.', 'TEACHER LETTER DRINK'),
    ('She is drinking clothes.', 'SHE CLOTHES DRINK'),
    ("I don't drink an apple.", 'I APPLE DRINK NOT'),
    ('We are cooking food.', 'WE FOOD COOK'),
    ('They are liking a computer.', 'THEY COMPUTER LIKE'),
    ('He ate water this morning.', 'MORNING HE WATER EAT FINISH'),
    ('He will read a mango next week.', 'NEXT WEEK HE MANGO READ WILL'),
    ('She bought a book this morning.', 'MORNING SHE BOOK BUY FINISH'),
    ('We are driving milk.', 'WE MILK DRIVE'),
    ('She is reading clothes.', 'SHE CLOTHES READ'),
    ('Why did the girl take an apple?', 'GIRL APPLE TAKE WHY'),
    ('She drank an apple yesterday.', 'YESTERDAY SHE APPLE DRINK FINISH'),
    ('She is buying a book.', 'SHE BOOK BUY'),
    ('We is cooking water now.', 'NOW WE WATER COOK'),
    ('Where is my friend going with a mango?', 'MY FRIEND MANGO GO WHERE'),
    ('She will cook water tomorrow.', 'TOMORROW SHE WATER COOK WILL'),
    ('Where is we going with a car?', 'WE CAR GO WHERE'),
    ('They will buy a book next week.', 'NEXT WEEK THEY BOOK BUY WILL'),
    ('My father is liking a house.', 'MY FATHER HOUSE LIKE'),
    ("She doesn't buy a mango.", 'SHE MANGO BUY NOT'),
    ('We are cooking a phone.', 'WE PHONE COOK'),
    ('The doctor is eating a bird.', 'DOCTOR BIRD EAT'),
    ('I is eating food today.', 'TODAY I FOOD EAT'),
    ('He is wanting water.', 'HE WATER WANT'),
    ('The teacher is eating a computer.', 'TEACHER COMPUTER EAT'),
    ('The teacher is cleaning a bird.', 'TEACHER BIRD CLEAN'),
    ('She is giving school.', 'SHE SCHOOL GIVE'),
    ('They is reading a book today.', 'TODAY THEY BOOK READ'),
    ('He is liking money.', 'HE MONEY LIKE'),
    ('We are wanting a book.', 'WE BOOK WANT'),
    ('My father is cooking an apple.', 'MY FATHER APPLE COOK'),
    ('He bought an apple yesterday.', 'YESTERDAY HE APPLE BUY FINISH'),
    ('I am making water.', 'I WATER MAKE'),
    ("I don't drink a bird.", 'I BIRD DRINK NOT'),
    ('I am drinking a computer.', 'I COMPUTER DRINK'),
    ("The teacher doesn't read a house.", 'TEACHER HOUSE READ NOT'),
    ('They are drinking a book.', 'THEY BOOK DRINK'),
    ('They cooked water yesterday.', 'YESTERDAY THEY WATER COOK FINISH'),
    ('Where is the girl going with water?', 'GIRL WATER GO WHERE'),
    ('The girl is making a computer.', 'GIRL COMPUTER MAKE'),
    ('The doctor is cooking a book.', 'DOCTOR BOOK COOK'),
    ('My father is cooking a computer.', 'MY FATHER COMPUTER COOK'),
    ("The girl doesn't eat a computer.", 'GIRL COMPUTER EAT NOT'),
    ('My father is wanting school.', 'MY FATHER SCHOOL WANT'),
    ('He is reading an apple tonight.', 'NIGHT HE APPLE READ'),
    ('I is cooking a mango tonight.', 'NIGHT I MANGO COOK'),
    ('She is buying a mango.', 'SHE MANGO BUY'),
    ('I am writing water.', 'I WATER WRITE'),
    ('They ate a book this morning.', 'MORNING THEY BOOK EAT FINISH'),
    ('They are driving a computer.', 'THEY COMPUTER DRIVE'),
    ('I am writing milk.', 'I MILK WRITE'),
    ('Why did the doctor take a computer?', 'DOCTOR COMPUTER TAKE WHY'),
    ("The girl doesn't buy food.", 'GIRL FOOD BUY NOT'),
    ('My father is buying school.', 'MY FATHER SCHOOL BUY'),
    ('I is drinking an apple tonight.', 'NIGHT I APPLE DRINK'),
    ("He doesn't cook an apple.", 'HE APPLE COOK NOT'),
    ('We are buying a computer.', 'WE COMPUTER BUY'),
    ('I will eat a book tomorrow.', 'TOMORROW I BOOK EAT WILL'),
    ('He is cleaning a computer.', 'HE COMPUTER CLEAN'),
    ("My father doesn't read water.", 'MY FATHER WATER READ NOT'),
    ('The doctor is liking a car.', 'DOCTOR CAR LIKE'),
    ('She will read food next week.', 'NEXT WEEK SHE FOOD READ WILL'),
    ('She ate an apple this morning.', 'MORNING SHE APPLE EAT FINISH'),
    ('He is driving an apple.', 'HE APPLE DRIVE'),
    ('The girl is liking an apple.', 'GIRL APPLE LIKE'),
    ("My friend doesn't cook an apple.", 'MY FRIEND APPLE COOK NOT'),
    ('The boy is taking a mango.', 'BOY MANGO TAKE'),
    ('He is driving a car.', 'HE CAR DRIVE'),
    ('We are wanting an apple.', 'WE APPLE WANT'),
    ('They are reading a computer.', 'THEY COMPUTER READ'),
    ("The doctor doesn't eat a book.", 'DOCTOR BOOK EAT NOT'),
    ('We are cleaning an apple.', 'WE APPLE CLEAN'),
    ('She will eat food next week.', 'NEXT WEEK SHE FOOD EAT WILL'),
    ('My mother is giving a mango.', 'MY MOTHER MANGO GIVE'),
    ('The boy is giving a car.', 'BOY CAR GIVE'),
    ('They are making money.', 'THEY MONEY MAKE'),
    ("The teacher doesn't read a car.", 'TEACHER CAR READ NOT'),
    ('Why did she take a computer?', 'SHE COMPUTER TAKE WHY'),
    ('The girl is making school.', 'GIRL SCHOOL MAKE'),
    ('The doctor is making school.', 'DOCTOR SCHOOL MAKE'),
    ("My mother doesn't read money.", 'MY MOTHER MONEY READ NOT'),
    ('The girl is giving milk.', 'GIRL MILK GIVE'),
    ('I is cooking water today.', 'TODAY I WATER COOK'),
    ("My mother doesn't drink a letter.", 'MY MOTHER LETTER DRINK NOT'),
    ('She is seeing an apple.', 'SHE APPLE SEE'),
    ('The girl is cooking a computer.', 'GIRL COMPUTER COOK'),
    ('I is reading food today.', 'TODAY I FOOD READ'),
    ('The doctor is cleaning a mango.', 'DOCTOR MANGO CLEAN'),
    ('He is buying clothes.', 'HE CLOTHES BUY'),
    ('I am wanting school.', 'I SCHOOL WANT'),
    ('They will cook a mango next week.', 'NEXT WEEK THEY MANGO COOK WILL'),
    ('My father is seeing a bird.', 'MY FATHER BIRD SEE'),
    ('She is giving a computer.', 'SHE COMPUTER GIVE'),
    ('The girl is writing a book.', 'GIRL BOOK WRITE'),
    ('My friend is making a house.', 'MY FRIEND HOUSE MAKE'),
    ("The teacher doesn't read a mango.", 'TEACHER MANGO READ NOT'),
    ('The boy is taking a phone.', 'BOY PHONE TAKE'),
    ('I bought food yesterday.', 'YESTERDAY I FOOD BUY FINISH'),
    ('He is drinking school.', 'HE SCHOOL DRINK'),
    ("He doesn't buy food.", 'HE FOOD BUY NOT'),
    ("The boy doesn't eat a bird.", 'BOY BIRD EAT NOT'),
    ('The doctor is cooking a letter.', 'DOCTOR LETTER COOK'),
    ('He drank water last week.', 'PAST WEEK HE WATER DRINK FINISH'),
    ("We don't drink an apple.", 'WE APPLE DRINK NOT'),
    ('Why did we take a phone?', 'WE PHONE TAKE WHY'),
    ('The doctor is reading a computer.', 'DOCTOR COMPUTER READ'),
    ('I will buy water tomorrow.', 'TOMORROW I WATER BUY WILL'),
    ('The doctor is making money.', 'DOCTOR MONEY MAKE'),
    ('My father is seeing a phone.', 'MY FATHER PHONE SEE'),
    ("The girl doesn't cook money.", 'GIRL MONEY COOK NOT'),
    ('We read water this morning.', 'MORNING WE WATER READ FINISH'),
    ('My friend is cleaning a computer.', 'MY FRIEND COMPUTER CLEAN'),
    ('She is liking school.', 'SHE SCHOOL LIKE'),
    ('My friend is buying school.', 'MY FRIEND SCHOOL BUY'),
    ('They cooked a mango last week.', 'PAST WEEK THEY MANGO COOK FINISH'),
    ('They is eating a book today.', 'TODAY THEY BOOK EAT'),
    ('The doctor is drinking a letter.', 'DOCTOR LETTER DRINK'),
    ("My mother doesn't cook a computer.", 'MY MOTHER COMPUTER COOK NOT'),
    ("The doctor doesn't buy a phone.", 'DOCTOR PHONE BUY NOT'),
    ('The teacher is wanting a letter.', 'TEACHER LETTER WANT'),
    ('We will cook food next week.', 'NEXT WEEK WE FOOD COOK WILL'),
    ("I don't cook money.", 'I MONEY COOK NOT'),
    ('My friend is buying clothes.', 'MY FRIEND CLOTHES BUY'),
    ('She is eating a phone.', 'SHE PHONE EAT'),
    ('He is giving a house.', 'HE HOUSE GIVE'),
    ('Why did they take a letter?', 'THEY LETTER TAKE WHY'),
    ('I am cleaning a letter.', 'I LETTER CLEAN'),
    ('Where is the doctor going with a phone?', 'DOCTOR PHONE GO WHERE'),
    ('They are washing a house.', 'THEY HOUSE WASH'),
    ('He is cooking a mango.', 'HE MANGO COOK'),
    ('We are reading an apple.', 'WE APPLE READ'),
    ('Why did my friend take milk?', 'MY FRIEND MILK TAKE WHY'),
    ('They is buying an apple now.', 'NOW THEY APPLE BUY'),
    ('The girl is giving money.', 'GIRL MONEY GIVE'),
    ('He ate a book this morning.', 'MORNING HE BOOK EAT FINISH'),
    ('Why did the doctor take food?', 'DOCTOR FOOD TAKE WHY'),
    ('Where is he going with water?', 'HE WATER GO WHERE'),
    ('He is washing school.', 'HE SCHOOL WASH'),
    ('My father is making a letter.', 'MY FATHER LETTER MAKE'),
    ('My friend is making money.', 'MY FRIEND MONEY MAKE'),
    ('My friend is seeing a computer.', 'MY FRIEND COMPUTER SEE'),
    ('Why did the girl take a letter?', 'GIRL LETTER TAKE WHY'),
    ('The doctor is buying a phone.', 'DOCTOR PHONE BUY'),
    ('She is drinking food now.', 'NOW SHE FOOD DRINK'),
    ('The teacher is writing school.', 'TEACHER SCHOOL WRITE'),
    ('She is seeing a house.', 'SHE HOUSE SEE'),
    ("The girl doesn't eat water.", 'GIRL WATER EAT NOT'),
    ('Why did the girl take a car?', 'GIRL CAR TAKE WHY'),
    ('My father is writing a computer.', 'MY FATHER COMPUTER WRITE'),
    ('We read water last week.', 'PAST WEEK WE WATER READ FINISH'),
    ('They are taking clothes.', 'THEY CLOTHES TAKE'),
    ('She is washing water.', 'SHE WATER WASH'),
    ('She cooked an apple yesterday.', 'YESTERDAY SHE APPLE COOK FINISH'),
    ('Why did the girl take money?', 'GIRL MONEY TAKE WHY'),
    ('He is reading a car.', 'HE CAR READ'),
    ('I am writing a letter.', 'I LETTER WRITE'),
    ('Where is they going with a house?', 'THEY HOUSE GO WHERE'),
    ('We bought a book yesterday.', 'YESTERDAY WE BOOK BUY FINISH'),
    ("The doctor doesn't buy a house.", 'DOCTOR HOUSE BUY NOT'),
    ('The girl is seeing a phone.', 'GIRL PHONE SEE'),
    ('Where is we going with a phone?', 'WE PHONE GO WHERE'),
    ('They is drinking a book tonight.', 'NIGHT THEY BOOK DRINK'),
    ('He cooked water this morning.', 'MORNING HE WATER COOK FINISH'),
    ('Where is we going with a mango?', 'WE MANGO GO WHERE'),
    ('The teacher is giving a car.', 'TEACHER CAR GIVE'),
    ("They don't eat milk.", 'THEY MILK EAT NOT'),
    ('The doctor is seeing a computer.', 'DOCTOR COMPUTER SEE'),
    ("She doesn't drink a phone.", 'SHE PHONE DRINK NOT'),
    ('The girl is driving water.', 'GIRL WATER DRIVE'),
    ('My father is reading water.', 'MY FATHER WATER READ'),
    ('We is reading a book today.', 'TODAY WE BOOK READ'),
    ('The teacher is buying water.', 'TEACHER WATER BUY'),
    ('We will buy food tomorrow.', 'TOMORROW WE FOOD BUY WILL'),
    ('Where is my father going with water?', 'MY FATHER WATER GO WHERE'),
    ("We don't buy money.", 'WE MONEY BUY NOT'),
    ('My father is drinking a book.', 'MY FATHER BOOK DRINK'),
    ('He read an apple yesterday.', 'YESTERDAY HE APPLE READ FINISH'),
    ('We are taking a letter.', 'WE LETTER TAKE'),
    ('She is seeing money.', 'SHE MONEY SEE'),
    ('What do we want?', 'WE WANT WHAT'),
    ('The boy is cooking money.', 'BOY MONEY COOK'),
    ('She bought water this morning.', 'MORNING SHE WATER BUY FINISH'),
    ("The doctor doesn't buy clothes.", 'DOCTOR CLOTHES BUY NOT'),
    ('They are drinking milk.', 'THEY MILK DRINK'),
    ('He is wanting clothes.', 'HE CLOTHES WANT'),
    ('She is buying a bird.', 'SHE BIRD BUY'),
    ('The teacher is cleaning an apple.', 'TEACHER APPLE CLEAN'),
    ('My mother is washing milk.', 'MY MOTHER MILK WASH'),
    ('The doctor is writing a phone.', 'DOCTOR PHONE WRITE'),
    ('The girl is cooking money.', 'GIRL MONEY COOK'),
    ('I am writing a house.', 'I HOUSE WRITE'),
    ("They don't buy milk.", 'THEY MILK BUY NOT'),
    ('We are eating milk.', 'WE MILK EAT'),
    ('My mother is cooking an apple.', 'MY MOTHER APPLE COOK'),
    ('They bought an apple this morning.', 'MORNING THEY APPLE BUY FINISH'),
    ('I am buying food.', 'I FOOD BUY'),
    ('I is eating a mango tonight.', 'NIGHT I MANGO EAT'),
    ('My mother is giving a phone.', 'MY MOTHER PHONE GIVE'),
    ('I am writing a bird.', 'I BIRD WRITE'),
    ('Why did she take water?', 'SHE WATER TAKE WHY'),
    ("The boy doesn't drink a computer.", 'BOY COMPUTER DRINK NOT'),
    ('He is making a car.', 'HE CAR MAKE'),
    ('I am wanting a mango.', 'I MANGO WANT'),
    ('She bought food yesterday.', 'YESTERDAY SHE FOOD BUY FINISH'),
    ('The girl is washing a mango.', 'GIRL MANGO WASH'),
    ("She doesn't drink money.", 'SHE MONEY DRINK NOT'),
    ('Why did we take a book?', 'WE BOOK TAKE WHY'),
    ('They ate a mango yesterday.', 'YESTERDAY THEY MANGO EAT FINISH'),
    ('He drank food last week.', 'PAST WEEK HE FOOD DRINK FINISH'),
    ('Where is they going with money?', 'THEY MONEY GO WHERE'),
    ('We bought an apple this morning.', 'MORNING WE APPLE BUY FINISH'),
    ('They are making water.', 'THEY WATER MAKE'),
    ('They bought an apple last week.', 'PAST WEEK THEY APPLE BUY FINISH'),
    ('My mother is washing a mango.', 'MY MOTHER MANGO WASH'),
    ('We read an apple yesterday.', 'YESTERDAY WE APPLE READ FINISH'),
    ("My father doesn't cook a mango.", 'MY FATHER MANGO COOK NOT'),
    ('The doctor is buying clothes.', 'DOCTOR CLOTHES BUY'),
    ('My father is wanting food.', 'MY FATHER FOOD WANT'),
    ('She cooked water this morning.', 'MORNING SHE WATER COOK FINISH'),
    ('We drank a mango last week.', 'PAST WEEK WE MANGO DRINK FINISH'),
    ('The teacher is cooking a phone.', 'TEACHER PHONE COOK'),
    ('My friend is taking a letter.', 'MY FRIEND LETTER TAKE'),
    ('He is cooking a book.', 'HE BOOK COOK'),
    ("My friend doesn't cook food.", 'MY FRIEND FOOD COOK NOT'),
    ('My mother is cleaning a bird.', 'MY MOTHER BIRD CLEAN'),
    ('They are driving a bird.', 'THEY BIRD DRIVE'),
    ('I am wanting a phone.', 'I PHONE WANT'),
    ("I don't eat a mango.", 'I MANGO EAT NOT'),
    ("The boy doesn't cook water.", 'BOY WATER COOK NOT'),
    ('Why did the doctor take school?', 'DOCTOR SCHOOL TAKE WHY'),
    ('They are cleaning a mango.', 'THEY MANGO CLEAN'),
    ("The girl doesn't eat clothes.", 'GIRL CLOTHES EAT NOT'),
    ('We is reading water now.', 'NOW WE WATER READ'),
    ('They are giving money.', 'THEY MONEY GIVE'),
    ('My father is buying a car.', 'MY FATHER CAR BUY'),
    ("He doesn't buy a letter.", 'HE LETTER BUY NOT'),
    ('My mother is cleaning a letter.', 'MY MOTHER LETTER CLEAN'),
    ("The boy doesn't buy a book.", 'BOY BOOK BUY NOT'),
    ('They is eating an apple today.', 'TODAY THEY APPLE EAT'),
    ('The boy is cleaning school.', 'BOY SCHOOL CLEAN'),
    ('They is reading an apple today.', 'TODAY THEY APPLE READ'),
    ('They will drink water next week.', 'NEXT WEEK THEY WATER DRINK WILL'),
    ('The doctor is eating a letter.', 'DOCTOR LETTER EAT'),
    ('My father is liking money.', 'MY FATHER MONEY LIKE'),
    ('The boy is reading a car.', 'BOY CAR READ'),
    ('He is cooking a book tonight.', 'NIGHT HE BOOK COOK'),
    ('The boy is making money.', 'BOY MONEY MAKE'),
    ('They is reading an apple tonight.', 'NIGHT THEY APPLE READ'),
    ('The girl is making clothes.', 'GIRL CLOTHES MAKE'),
    ('We are eating water.', 'WE WATER EAT'),
    ('Where is the boy going with school?', 'BOY SCHOOL GO WHERE'),
    ('I is reading a mango today.', 'TODAY I MANGO READ'),
    ('She is buying water now.', 'NOW SHE WATER BUY'),
    ('He will drink a book next week.', 'NEXT WEEK HE BOOK DRINK WILL'),
    ("We don't drink clothes.", 'WE CLOTHES DRINK NOT'),
    ('Why did we take water?', 'WE WATER TAKE WHY'),
    ('Where is the teacher going with a letter?', 'TEACHER LETTER GO WHERE'),
    ('He is cooking an apple.', 'HE APPLE COOK'),
    ('He is writing a phone.', 'HE PHONE WRITE'),
    ('We are liking a car.', 'WE CAR LIKE'),
    ('My friend is eating a house.', 'MY FRIEND HOUSE EAT'),
    ('The boy is buying money.', 'BOY MONEY BUY'),
    ("They don't read an apple.", 'THEY APPLE READ NOT'),
    ('My mother is cooking a house.', 'MY MOTHER HOUSE COOK'),
    ('The teacher is cooking money.', 'TEACHER MONEY COOK'),
    ("We don't eat a computer.", 'WE COMPUTER EAT NOT'),
    ('He cooked water yesterday.', 'YESTERDAY HE WATER COOK FINISH'),
    ('We will eat a book tomorrow.', 'TOMORROW WE BOOK EAT WILL'),
    ('I is drinking water now.', 'NOW I WATER DRINK'),
    ('She is cooking food.', 'SHE FOOD COOK'),
    ('The boy is liking a phone.', 'BOY PHONE LIKE'),
    ('They will cook water tomorrow.', 'TOMORROW THEY WATER COOK WILL'),
    ('The teacher is buying a house.', 'TEACHER HOUSE BUY'),
    ('My friend is giving a phone.', 'MY FRIEND PHONE GIVE'),
    ('They is drinking a mango tonight.', 'NIGHT THEY MANGO DRINK'),
    ('I am buying a phone.', 'I PHONE BUY'),
    ('The doctor is driving milk.', 'DOCTOR MILK DRIVE'),
    ('He is wanting milk.', 'HE MILK WANT'),
    ('The boy is cleaning clothes.', 'BOY CLOTHES CLEAN'),
    ('He will buy a book tomorrow.', 'TOMORROW HE BOOK BUY WILL'),
    ('They are writing a letter.', 'THEY LETTER WRITE'),
    ('The girl is seeing a computer.', 'GIRL COMPUTER SEE'),
    ('She will eat a book tomorrow.', 'TOMORROW SHE BOOK EAT WILL'),
    ("My mother doesn't drink milk.", 'MY MOTHER MILK DRINK NOT'),
    ('The doctor is driving a computer.', 'DOCTOR COMPUTER DRIVE'),
    ('I is buying a mango now.', 'NOW I MANGO BUY'),
    ('She bought a book yesterday.', 'YESTERDAY SHE BOOK BUY FINISH'),
    ('They are cleaning milk.', 'THEY MILK CLEAN'),
    ('Where is my mother going with a house?', 'MY MOTHER HOUSE GO WHERE'),
    ('The teacher is eating water.', 'TEACHER WATER EAT'),
    ('My mother is cooking clothes.', 'MY MOTHER CLOTHES COOK'),
    ('I is drinking a mango now.', 'NOW I MANGO DRINK'),
    ('She drank a mango yesterday.', 'YESTERDAY SHE MANGO DRINK FINISH'),
    ('The doctor is wanting a mango.', 'DOCTOR MANGO WANT'),
    ("The teacher doesn't eat a house.", 'TEACHER HOUSE EAT NOT'),
    ('They are seeing a bird.', 'THEY BIRD SEE'),
    ("My friend doesn't eat a bird.", 'MY FRIEND BIRD EAT NOT'),
    ("My friend doesn't drink a computer.", 'MY FRIEND COMPUTER DRINK NOT'),
    ('The teacher is liking a mango.', 'TEACHER MANGO LIKE'),
    ('I am driving a book.', 'I BOOK DRIVE'),
    ("He doesn't drink money.", 'HE MONEY DRINK NOT'),
    ('He is reading a bird.', 'HE BIRD READ'),
    ('The girl is washing money.', 'GIRL MONEY WASH'),
    ('They ate an apple last week.', 'PAST WEEK THEY APPLE EAT FINISH'),
    ('My friend is buying a letter.', 'MY FRIEND LETTER BUY'),
    ('The girl is giving water.', 'GIRL WATER GIVE'),
    ("The girl doesn't cook a car.", 'GIRL CAR COOK NOT'),
    ('They are taking a car.', 'THEY CAR TAKE'),
    ('He is cooking a car.', 'HE CAR COOK'),
    ('I am making a car.', 'I CAR MAKE'),
    ('They will cook food tomorrow.', 'TOMORROW THEY FOOD COOK WILL'),
    ('I am taking a computer.', 'I COMPUTER TAKE'),
    ("My father doesn't read a car.", 'MY FATHER CAR READ NOT'),
    ('They are wanting water.', 'THEY WATER WANT'),
    ("She doesn't read clothes.", 'SHE CLOTHES READ NOT'),
    ('I am eating an apple.', 'I APPLE EAT'),
    ('She is wanting water.', 'SHE WATER WANT'),
    ('They will cook food next week.', 'NEXT WEEK THEY FOOD COOK WILL'),
    ('I am writing a book.', 'I BOOK WRITE'),
    ('We cooked water yesterday.', 'YESTERDAY WE WATER COOK FINISH'),
    ('The teacher is eating school.', 'TEACHER SCHOOL EAT'),
    ('He is making an apple.', 'HE APPLE MAKE'),
    ("The doctor doesn't read school.", 'DOCTOR SCHOOL READ NOT'),
    ('I am drinking a letter.', 'I LETTER DRINK'),
    ("The doctor doesn't drink school.", 'DOCTOR SCHOOL DRINK NOT'),
    ('They are washing a letter.', 'THEY LETTER WASH'),
    ('My father is making clothes.', 'MY FATHER CLOTHES MAKE'),
    ('Where is they going with food?', 'THEY FOOD GO WHERE'),
    ('They are cooking a bird.', 'THEY BIRD COOK'),
    ("The boy doesn't cook a bird.", 'BOY BIRD COOK NOT'),
    ('The teacher is writing a book.', 'TEACHER BOOK WRITE'),
    ('My friend is wanting a car.', 'MY FRIEND CAR WANT'),
    ('The girl is drinking a car.', 'GIRL CAR DRINK'),
    ('I is eating food now.', 'NOW I FOOD EAT'),
    ("The doctor doesn't eat a car.", 'DOCTOR CAR EAT NOT'),
    ('He read an apple this morning.', 'MORNING HE APPLE READ FINISH'),
    ('We is reading a book now.', 'NOW WE BOOK READ'),
    ('The boy is seeing a house.', 'BOY HOUSE SEE'),
    ('We are seeing food.', 'WE FOOD SEE'),
    ('My mother is drinking a bird.', 'MY MOTHER BIRD DRINK'),
    ('He is washing money.', 'HE MONEY WASH'),
    ('I am wanting a bird.', 'I BIRD WANT'),
    ('She is writing a letter.', 'SHE LETTER WRITE'),
    ('They drank a mango this morning.', 'MORNING THEY MANGO DRINK FINISH'),
    ('We cooked a book yesterday.', 'YESTERDAY WE BOOK COOK FINISH'),
    ("The girl doesn't buy an apple.", 'GIRL APPLE BUY NOT'),
    ("He doesn't eat milk.", 'HE MILK EAT NOT'),
    ('The boy is reading a bird.', 'BOY BIRD READ'),
    ('We are giving food.', 'WE FOOD GIVE'),
    ('He is cooking a mango tonight.', 'NIGHT HE MANGO COOK'),
    ('I am washing a bird.', 'I BIRD WASH'),
    ("He doesn't drink a phone.", 'HE PHONE DRINK NOT'),
    ('They are wanting school.', 'THEY SCHOOL WANT'),
    ("The doctor doesn't buy a mango.", 'DOCTOR MANGO BUY NOT'),
    ('My friend is liking a computer.', 'MY FRIEND COMPUTER LIKE'),
    ('The teacher is eating money.', 'TEACHER MONEY EAT'),
    ('He is buying a book now.', 'NOW HE BOOK BUY'),
    ('She is making a letter.', 'SHE LETTER MAKE'),
    ("The doctor doesn't eat clothes.", 'DOCTOR CLOTHES EAT NOT'),
    ("The boy doesn't drink money.", 'BOY MONEY DRINK NOT'),
    ('The boy is giving an apple.', 'BOY APPLE GIVE'),
    ('We are washing a book.', 'WE BOOK WASH'),
    ('My friend is buying milk.', 'MY FRIEND MILK BUY'),
    ("The teacher doesn't read food.", 'TEACHER FOOD READ NOT'),
    ('My friend is writing clothes.', 'MY FRIEND CLOTHES WRITE'),
    ('They are cleaning school.', 'THEY SCHOOL CLEAN'),
    ('I am wanting water.', 'I WATER WANT'),
    ('He will buy a mango next week.', 'NEXT WEEK HE MANGO BUY WILL'),
    ('Where is the boy going with a mango?', 'BOY MANGO GO WHERE'),
    ('My mother is drinking a car.', 'MY MOTHER CAR DRINK'),
    ('I is drinking food today.', 'TODAY I FOOD DRINK'),
    ("The boy doesn't drink a book.", 'BOY BOOK DRINK NOT'),
    ('The girl is liking a car.', 'GIRL CAR LIKE'),
    ("I don't drink a house.", 'I HOUSE DRINK NOT'),
    ("She doesn't cook a mango.", 'SHE MANGO COOK NOT'),
    ("My father doesn't buy a letter.", 'MY FATHER LETTER BUY NOT'),
    ('Where is the boy going with a computer?', 'BOY COMPUTER GO WHERE'),
    ('They are reading a car.', 'THEY CAR READ'),
    ('We drank an apple this morning.', 'MORNING WE APPLE DRINK FINISH'),
    ('She is cleaning food.', 'SHE FOOD CLEAN'),
    ('The girl is eating a phone.', 'GIRL PHONE EAT'),
    ("The doctor doesn't drink a computer.", 'DOCTOR COMPUTER DRINK NOT'),
    ("The boy doesn't read a mango.", 'BOY MANGO READ NOT'),
    ('My mother is drinking a mango.', 'MY MOTHER MANGO DRINK'),
    ('The boy is buying a car.', 'BOY CAR BUY'),
    ('My friend is taking milk.', 'MY FRIEND MILK TAKE'),
    ('The boy is driving milk.', 'BOY MILK DRIVE'),
    ('They drank water last week.', 'PAST WEEK THEY WATER DRINK FINISH'),
    ("The boy doesn't buy clothes.", 'BOY CLOTHES BUY NOT'),
    ("My father doesn't buy food.", 'MY FATHER FOOD BUY NOT'),
    ('The teacher is drinking a car.', 'TEACHER CAR DRINK'),
    ('She ate a book yesterday.', 'YESTERDAY SHE BOOK EAT FINISH'),
    ('My mother is eating clothes.', 'MY MOTHER CLOTHES EAT'),
    ('We drank an apple yesterday.', 'YESTERDAY WE APPLE DRINK FINISH'),
    ('We are drinking money.', 'WE MONEY DRINK'),
    ("We don't buy an apple.", 'WE APPLE BUY NOT'),
    ("We don't drink a car.", 'WE CAR DRINK NOT'),
    ("My father doesn't eat a bird.", 'MY FATHER BIRD EAT NOT'),
    ('They cooked a mango yesterday.', 'YESTERDAY THEY MANGO COOK FINISH'),
    ('He is liking a book.', 'HE BOOK LIKE'),
    ('The teacher is drinking water.', 'TEACHER WATER DRINK'),
    ('Why did the teacher take food?', 'TEACHER FOOD TAKE WHY'),
    ('He is cleaning water.', 'HE WATER CLEAN'),
    ('They are cleaning an apple.', 'THEY APPLE CLEAN'),
    ('My friend is drinking a house.', 'MY FRIEND HOUSE DRINK'),
    ('She is eating water now.', 'NOW SHE WATER EAT'),
    ('I am washing water.', 'I WATER WASH'),
    ('The doctor is making a letter.', 'DOCTOR LETTER MAKE'),
    ('My mother is eating a house.', 'MY MOTHER HOUSE EAT'),
    ('My father is wanting water.', 'MY FATHER WATER WANT'),
    ('My mother is taking food.', 'MY MOTHER FOOD TAKE'),
    ('Why did the girl take a phone?', 'GIRL PHONE TAKE WHY'),
    ('He is reading a mango today.', 'TODAY HE MANGO READ'),
    ('My mother is wanting a bird.', 'MY MOTHER BIRD WANT'),
    ('He will cook food tomorrow.', 'TOMORROW HE FOOD COOK WILL'),
    ('I am seeing a car.', 'I CAR SEE'),
    ('My father is taking a bird.', 'MY FATHER BIRD TAKE'),
    ('They is buying a mango tonight.', 'NIGHT THEY MANGO BUY'),
    ("The boy doesn't drink a letter.", 'BOY LETTER DRINK NOT'),
    ('My friend is reading clothes.', 'MY FRIEND CLOTHES READ'),
    ('My friend is reading school.', 'MY FRIEND SCHOOL READ'),
    ("My father doesn't cook a phone.", 'MY FATHER PHONE COOK NOT'),
    ("He doesn't drink clothes.", 'HE CLOTHES DRINK NOT'),
    ("We don't drink a mango.", 'WE MANGO DRINK NOT'),
    ("He doesn't read a bird.", 'HE BIRD READ NOT'),
    ('My friend is reading a mango.', 'MY FRIEND MANGO READ'),
    ('He is taking a car.', 'HE CAR TAKE'),
    ("My father doesn't cook water.", 'MY FATHER WATER COOK NOT'),
    ("My friend doesn't read a mango.", 'MY FRIEND MANGO READ NOT'),
    ("She doesn't read milk.", 'SHE MILK READ NOT'),
    ("The doctor doesn't eat school.", 'DOCTOR SCHOOL EAT NOT'),
    ('He read a book yesterday.', 'YESTERDAY HE BOOK READ FINISH'),
    ('We are cleaning food.', 'WE FOOD CLEAN'),
    ('The teacher is wanting food.', 'TEACHER FOOD WANT'),
    ('The teacher is giving an apple.', 'TEACHER APPLE GIVE'),
    ('I read water this morning.', 'MORNING I WATER READ FINISH'),
    ("The teacher doesn't drink water.", 'TEACHER WATER DRINK NOT'),
    ('The girl is washing a house.', 'GIRL HOUSE WASH'),
    ('I am driving milk.', 'I MILK DRIVE'),
    ("We don't drink money.", 'WE MONEY DRINK NOT'),
    ('I cooked water yesterday.', 'YESTERDAY I WATER COOK FINISH'),
    ('The boy is cleaning milk.', 'BOY MILK CLEAN'),
    ('She drank water this morning.', 'MORNING SHE WATER DRINK FINISH'),
    ("The boy doesn't cook school.", 'BOY SCHOOL COOK NOT'),
    ('The teacher is making an apple.', 'TEACHER APPLE MAKE'),
    ('The doctor is reading an apple.', 'DOCTOR APPLE READ'),
    ('He is drinking a mango today.', 'TODAY HE MANGO DRINK'),
    ('He is writing a computer.', 'HE COMPUTER WRITE'),
    ('I is buying food now.', 'NOW I FOOD BUY'),
    ('They are drinking a computer.', 'THEY COMPUTER DRINK'),
    ('The doctor is driving a bird.', 'DOCTOR BIRD DRIVE'),
    ('The doctor is driving clothes.', 'DOCTOR CLOTHES DRIVE'),
    ('The boy is wanting a car.', 'BOY CAR WANT'),
    ('They is eating a mango tonight.', 'NIGHT THEY MANGO EAT'),
    ("The boy doesn't buy a house.", 'BOY HOUSE BUY NOT'),
    ('He is reading water today.', 'TODAY HE WATER READ'),
    ('We ate an apple yesterday.', 'YESTERDAY WE APPLE EAT FINISH'),
    ("They don't read school.", 'THEY SCHOOL READ NOT'),
    ('My friend is washing school.', 'MY FRIEND SCHOOL WASH'),
    ('He is eating a mango today.', 'TODAY HE MANGO EAT'),
    ('He is liking school.', 'HE SCHOOL LIKE'),
    ('The boy is drinking money.', 'BOY MONEY DRINK'),
    ('She is making a book.', 'SHE BOOK MAKE'),
    ("The teacher doesn't buy food.", 'TEACHER FOOD BUY NOT'),
    ("I don't buy food.", 'I FOOD BUY NOT'),
    ('She is seeing a computer.', 'SHE COMPUTER SEE'),
    ('She is cleaning money.', 'SHE MONEY CLEAN'),
    ("My friend doesn't buy a book.", 'MY FRIEND BOOK BUY NOT'),
    ("The girl doesn't drink water.", 'GIRL WATER DRINK NOT'),
    ('She is wanting milk.', 'SHE MILK WANT'),
    ('He will buy water tomorrow.', 'TOMORROW HE WATER BUY WILL'),
    ('The boy is making a phone.', 'BOY PHONE MAKE'),
    ('We cooked food last week.', 'PAST WEEK WE FOOD COOK FINISH'),
    ('The teacher is wanting a house.', 'TEACHER HOUSE WANT'),
    ('I will buy a mango tomorrow.', 'TOMORROW I MANGO BUY WILL'),
    ('My mother is driving water.', 'MY MOTHER WATER DRIVE'),
    ('The boy is cleaning a computer.', 'BOY COMPUTER CLEAN'),
    ('We are giving a car.', 'WE CAR GIVE'),
    ('He is cooking food.', 'HE FOOD COOK'),
    ('My friend is washing a mango.', 'MY FRIEND MANGO WASH'),
    ('He will cook a mango next week.', 'NEXT WEEK HE MANGO COOK WILL'),
    ("The girl doesn't buy a car.", 'GIRL CAR BUY NOT'),
    ('Where is my mother going with a bird?', 'MY MOTHER BIRD GO WHERE'),
    ('The boy is drinking school.', 'BOY SCHOOL DRINK'),
    ('We are writing a house.', 'WE HOUSE WRITE'),
    ('They are washing clothes.', 'THEY CLOTHES WASH'),
    ("The doctor doesn't cook a mango.", 'DOCTOR MANGO COOK NOT'),
    ('The boy is washing milk.', 'BOY MILK WASH'),
    ('The girl is wanting a bird.', 'GIRL BIRD WANT'),
    ('The girl is liking a letter.', 'GIRL LETTER LIKE'),
    ('My friend is driving a book.', 'MY FRIEND BOOK DRIVE'),
    ('I am drinking a car.', 'I CAR DRINK'),
    ('The girl is cleaning a phone.', 'GIRL PHONE CLEAN'),
    ('We are buying school.', 'WE SCHOOL BUY'),
    ('She is driving water.', 'SHE WATER DRIVE'),
    ('Why did the doctor take a house?', 'DOCTOR HOUSE TAKE WHY'),
    ('He drank a mango this morning.', 'MORNING HE MANGO DRINK FINISH'),
    ('I am drinking money.', 'I MONEY DRINK'),
    ('She is washing a house.', 'SHE HOUSE WASH'),
    ('The teacher is liking a bird.', 'TEACHER BIRD LIKE'),
    ('My father is giving a letter.', 'MY FATHER LETTER GIVE'),
    ('My mother is cleaning clothes.', 'MY MOTHER CLOTHES CLEAN'),
    ('We will buy water tomorrow.', 'TOMORROW WE WATER BUY WILL'),
    ('The girl is buying a mango.', 'GIRL MANGO BUY'),
    ('They are washing a computer.', 'THEY COMPUTER WASH'),
    ('Why did he take money?', 'HE MONEY TAKE WHY'),
    ('We cooked a book last week.', 'PAST WEEK WE BOOK COOK FINISH'),
    ("The girl doesn't cook a computer.", 'GIRL COMPUTER COOK NOT'),
    ('The teacher is cooking clothes.', 'TEACHER CLOTHES COOK'),
    ('The teacher is driving an apple.', 'TEACHER APPLE DRIVE'),
    ('I is drinking food tonight.', 'NIGHT I FOOD DRINK'),
    ('He will cook a book next week.', 'NEXT WEEK HE BOOK COOK WILL'),
    ('Why did i take a computer?', 'I COMPUTER TAKE WHY'),
    ('They are writing a bird.', 'THEY BIRD WRITE'),
    ('She will drink a book next week.', 'NEXT WEEK SHE BOOK DRINK WILL'),
    ('We are buying milk.', 'WE MILK BUY'),
    ('The boy is cooking a book.', 'BOY BOOK COOK'),
    ('The boy is liking school.', 'BOY SCHOOL LIKE'),
    ("My mother doesn't buy food.", 'MY MOTHER FOOD BUY NOT'),
    ("The girl doesn't read food.", 'GIRL FOOD READ NOT'),
    ('Why did the girl take a house?', 'GIRL HOUSE TAKE WHY'),
    ('Where is my mother going with a book?', 'MY MOTHER BOOK GO WHERE'),
    ('I will read a mango next week.', 'NEXT WEEK I MANGO READ WILL'),
    ('My friend is driving a mango.', 'MY FRIEND MANGO DRIVE'),
    ('They will read a mango tomorrow.', 'TOMORROW THEY MANGO READ WILL'),
    ("We don't cook money.", 'WE MONEY COOK NOT'),
    ('My friend is cooking a phone.', 'MY FRIEND PHONE COOK'),
    ('My friend is drinking food.', 'MY FRIEND FOOD DRINK'),
    ('She is seeing school.', 'SHE SCHOOL SEE'),
    ('She is cleaning a car.', 'SHE CAR CLEAN'),
    ('They are driving food.', 'THEY FOOD DRIVE'),
    ('The teacher is washing milk.', 'TEACHER MILK WASH'),
    ('We are taking a phone.', 'WE PHONE TAKE'),
    ("The boy doesn't read a car.", 'BOY CAR READ NOT'),
    ('They read a mango this morning.', 'MORNING THEY MANGO READ FINISH'),
    ('She is driving an apple.', 'SHE APPLE DRIVE'),
    ('He is driving a phone.', 'HE PHONE DRIVE'),
    ("She doesn't read school.", 'SHE SCHOOL READ NOT'),
    ('They are cleaning a car.', 'THEY CAR CLEAN'),
    ('The boy is wanting money.', 'BOY MONEY WANT'),
    ('The boy is liking a computer.', 'BOY COMPUTER LIKE'),
    ('The girl is drinking school.', 'GIRL SCHOOL DRINK'),
    ('My father is wanting a letter.', 'MY FATHER LETTER WANT'),
    ('I is cooking food today.', 'TODAY I FOOD COOK'),
    ('I bought a mango yesterday.', 'YESTERDAY I MANGO BUY FINISH'),
    ('We are giving a house.', 'WE HOUSE GIVE'),
    ('Where is she going with a car?', 'SHE CAR GO WHERE'),
    ('The girl is seeing a mango.', 'GIRL MANGO SEE'),
    ('The boy is making a bird.', 'BOY BIRD MAKE'),
    ('We are cooking school.', 'WE SCHOOL COOK'),
    ('My mother is driving an apple.', 'MY MOTHER APPLE DRIVE'),
    ('I am giving a mango.', 'I MANGO GIVE'),
    ('The boy is seeing money.', 'BOY MONEY SEE'),
    ('He will eat an apple tomorrow.', 'TOMORROW HE APPLE EAT WILL'),
    ('He will cook a mango tomorrow.', 'TOMORROW HE MANGO COOK WILL'),
    ('I will eat water next week.', 'NEXT WEEK I WATER EAT WILL'),
    ('I will cook a mango tomorrow.', 'TOMORROW I MANGO COOK WILL'),
    ('They is eating a mango now.', 'NOW THEY MANGO EAT'),
    ('I is reading food tonight.', 'NIGHT I FOOD READ'),
    ('Where is the girl going with a house?', 'GIRL HOUSE GO WHERE'),
    ('The teacher is wanting water.', 'TEACHER WATER WANT'),
    ('We ate water this morning.', 'MORNING WE WATER EAT FINISH'),
    ('He is buying water now.', 'NOW HE WATER BUY'),
    ('We is cooking an apple tonight.', 'NIGHT WE APPLE COOK'),
    ('The doctor is making a computer.', 'DOCTOR COMPUTER MAKE'),
    ("My mother doesn't buy money.", 'MY MOTHER MONEY BUY NOT'),
    ("My friend doesn't eat a book.", 'MY FRIEND BOOK EAT NOT'),
    ('She is giving a mango.', 'SHE MANGO GIVE'),
    ('We is eating a book now.', 'NOW WE BOOK EAT'),
    ('He is cooking water today.', 'TODAY HE WATER COOK'),
    ("They don't buy school.", 'THEY SCHOOL BUY NOT'),
    ('We bought water last week.', 'PAST WEEK WE WATER BUY FINISH'),
    ('She cooked water yesterday.', 'YESTERDAY SHE WATER COOK FINISH'),
    ('I am cleaning a bird.', 'I BIRD CLEAN'),
    ('The teacher is washing money.', 'TEACHER MONEY WASH'),
    ('My friend is giving water.', 'MY FRIEND WATER GIVE'),
    ('She is reading a book today.', 'TODAY SHE BOOK READ'),
    ('She will buy a mango tomorrow.', 'TOMORROW SHE MANGO BUY WILL'),
    ('She is eating an apple today.', 'TODAY SHE APPLE EAT'),
    ('Where is he going with an apple?', 'HE APPLE GO WHERE'),
    ("My mother doesn't cook a bird.", 'MY MOTHER BIRD COOK NOT'),
    ('I am liking a car.', 'I CAR LIKE'),
    ('What does he want?', 'HE WANT WHAT'),
    ('She is drinking a bird.', 'SHE BIRD DRINK'),
    ('The doctor is seeing water.', 'DOCTOR WATER SEE'),
    ('We are taking an apple.', 'WE APPLE TAKE'),
    ('They is cooking food now.', 'NOW THEY FOOD COOK'),
    ('The girl is washing a letter.', 'GIRL LETTER WASH'),
    ("We don't drink a house.", 'WE HOUSE DRINK NOT'),
    ('My friend is cooking water.', 'MY FRIEND WATER COOK'),
    ('They is cooking food today.', 'TODAY THEY FOOD COOK'),
    ('My friend is eating a letter.', 'MY FRIEND LETTER EAT'),
    ('She is writing milk.', 'SHE MILK WRITE'),
    ('They are wanting clothes.', 'THEY CLOTHES WANT'),
    ('The doctor is eating a house.', 'DOCTOR HOUSE EAT'),
    ('We will drink an apple tomorrow.', 'TOMORROW WE APPLE DRINK WILL'),
    ('The boy is washing clothes.', 'BOY CLOTHES WASH'),
    ("I don't cook an apple.", 'I APPLE COOK NOT'),
    ('I am cleaning a book.', 'I BOOK CLEAN'),
    ('They will drink a mango tomorrow.', 'TOMORROW THEY MANGO DRINK WILL'),
    ('The teacher is buying a computer.', 'TEACHER COMPUTER BUY'),
    ("The girl doesn't drink school.", 'GIRL SCHOOL DRINK NOT'),
    ("I don't eat milk.", 'I MILK EAT NOT'),
    ("The boy doesn't cook an apple.", 'BOY APPLE COOK NOT'),
    ("I don't drink a car.", 'I CAR DRINK NOT'),
    ('Where is he going with a letter?', 'HE LETTER GO WHERE'),
    ("We don't eat a bird.", 'WE BIRD EAT NOT'),
    ('They are making a book.', 'THEY BOOK MAKE'),
    ('They cooked food yesterday.', 'YESTERDAY THEY FOOD COOK FINISH'),
    ("The girl doesn't cook milk.", 'GIRL MILK COOK NOT'),
    ('I am taking a house.', 'I HOUSE TAKE'),
    ('They cooked an apple yesterday.', 'YESTERDAY THEY APPLE COOK FINISH'),
    ('We drank a mango yesterday.', 'YESTERDAY WE MANGO DRINK FINISH'),
    ('My friend is cleaning a bird.', 'MY FRIEND BIRD CLEAN'),
    ('She read food last week.', 'PAST WEEK SHE FOOD READ FINISH'),
    ('The boy is taking water.', 'BOY WATER TAKE'),
    ('I am eating a phone.', 'I PHONE EAT'),
    ('My mother is wanting a house.', 'MY MOTHER HOUSE WANT'),
    ("The boy doesn't buy a car.", 'BOY CAR BUY NOT'),
    ('We are writing a car.', 'WE CAR WRITE'),
    ('She is wanting money.', 'SHE MONEY WANT'),
    ('We is drinking a book today.', 'TODAY WE BOOK DRINK'),
    ('We is cooking a book tonight.', 'NIGHT WE BOOK COOK'),
    ("My friend doesn't drink a letter.", 'MY FRIEND LETTER DRINK NOT'),
    ('They are giving a phone.', 'THEY PHONE GIVE'),
    ('The doctor is wanting school.', 'DOCTOR SCHOOL WANT'),
    ('She drank a book this morning.', 'MORNING SHE BOOK DRINK FINISH'),
    ('He drank a book last week.', 'PAST WEEK HE BOOK DRINK FINISH'),
    ('I is buying water tonight.', 'NIGHT I WATER BUY'),
    ("The doctor doesn't eat food.", 'DOCTOR FOOD EAT NOT'),
    ('My father is liking food.', 'MY FATHER FOOD LIKE'),
    ("The doctor doesn't drink a bird.", 'DOCTOR BIRD DRINK NOT'),
    ('They are cooking a computer.', 'THEY COMPUTER COOK'),
    ("The girl doesn't drink money.", 'GIRL MONEY DRINK NOT'),
    ('Where is the boy going with clothes?', 'BOY CLOTHES GO WHERE'),
    ('I will eat food tomorrow.', 'TOMORROW I FOOD EAT WILL'),
    ('The doctor is eating school.', 'DOCTOR SCHOOL EAT'),
    ('We are liking a house.', 'WE HOUSE LIKE'),
    ('I is cooking a book now.', 'NOW I BOOK COOK'),
    ('Where is my father going with an apple?', 'MY FATHER APPLE GO WHERE'),
    ('The boy is taking an apple.', 'BOY APPLE TAKE'),
    ('He ate water last week.', 'PAST WEEK HE WATER EAT FINISH'),
    ("The teacher doesn't cook food.", 'TEACHER FOOD COOK NOT'),
    ('I am cooking a car.', 'I CAR COOK'),
    ('I am seeing milk.', 'I MILK SEE'),
    ('He is seeing money.', 'HE MONEY SEE'),
    ('She will read a book tomorrow.', 'TOMORROW SHE BOOK READ WILL'),
    ('I am making a book.', 'I BOOK MAKE'),
    ('The doctor is seeing a car.', 'DOCTOR CAR SEE'),
    ('My friend is giving a car.', 'MY FRIEND CAR GIVE'),
    ('I am buying water.', 'I WATER BUY'),
    ('She is buying an apple tonight.', 'NIGHT SHE APPLE BUY'),
    ("She doesn't cook a computer.", 'SHE COMPUTER COOK NOT'),
    ("I don't buy an apple.", 'I APPLE BUY NOT'),
    ('My friend is making a book.', 'MY FRIEND BOOK MAKE'),
    ("He doesn't eat a mango.", 'HE MANGO EAT NOT'),
    ('We are seeing school.', 'WE SCHOOL SEE'),
    ('My mother is wanting water.', 'MY MOTHER WATER WANT'),
    ("The girl doesn't read a bird.", 'GIRL BIRD READ NOT'),
    ("He doesn't eat food.", 'HE FOOD EAT NOT'),
    ('He will buy an apple next week.', 'NEXT WEEK HE APPLE BUY WILL'),
    ("My friend doesn't cook a car.", 'MY FRIEND CAR COOK NOT'),
    ('I am wanting a book.', 'I BOOK WANT'),
    ('They are reading school.', 'THEY SCHOOL READ'),
    ('My father is cleaning milk.', 'MY FATHER MILK CLEAN'),
    ('We is drinking a mango now.', 'NOW WE MANGO DRINK'),
    ('The teacher is writing food.', 'TEACHER FOOD WRITE'),
    ('They are cooking a car.', 'THEY CAR COOK'),
    ("He doesn't read money.", 'HE MONEY READ NOT'),
    ('She drank a mango last week.', 'PAST WEEK SHE MANGO DRINK FINISH'),
    ('We are liking a bird.', 'WE BIRD LIKE'),
    ("They don't read a book.", 'THEY BOOK READ NOT'),
    ('She is liking water.', 'SHE WATER LIKE'),
    ('I am seeing a bird.', 'I BIRD SEE'),
    ('Where is we going with milk?', 'WE MILK GO WHERE'),
    ('The teacher is drinking an apple.', 'TEACHER APPLE DRINK'),
    ("The teacher doesn't read school.", 'TEACHER SCHOOL READ NOT'),
    ('He is taking clothes.', 'HE CLOTHES TAKE'),
    ('They are cleaning a phone.', 'THEY PHONE CLEAN'),
    ("We don't eat a book.", 'WE BOOK EAT NOT'),
    ('They are liking clothes.', 'THEY CLOTHES LIKE'),
    ('He drank an apple last week.', 'PAST WEEK HE APPLE DRINK FINISH'),
    ("We don't read a house.", 'WE HOUSE READ NOT'),
    ('They is cooking water tonight.', 'NIGHT THEY WATER COOK'),
    ('I am wanting food.', 'I FOOD WANT'),
    ("He doesn't eat clothes.", 'HE CLOTHES EAT NOT'),
    ('My mother is washing a bird.', 'MY MOTHER BIRD WASH'),
    ("They don't read food.", 'THEY FOOD READ NOT'),
    ('The boy is driving an apple.', 'BOY APPLE DRIVE'),
    ("The teacher doesn't cook a letter.", 'TEACHER LETTER COOK NOT'),
    ('The teacher is reading food.', 'TEACHER FOOD READ'),
    ('Where is i going with water?', 'I WATER GO WHERE'),
    ('My mother is eating a computer.', 'MY MOTHER COMPUTER EAT'),
    ('My father is seeing a computer.', 'MY FATHER COMPUTER SEE'),
    ('She is giving food.', 'SHE FOOD GIVE'),
    ('My father is writing food.', 'MY FATHER FOOD WRITE'),
    ('The doctor is buying milk.', 'DOCTOR MILK BUY'),
    ('Why did my friend take a letter?', 'MY FRIEND LETTER TAKE WHY'),
    ("The teacher doesn't eat a bird.", 'TEACHER BIRD EAT NOT'),
    ('Where is the boy going with an apple?', 'BOY APPLE GO WHERE'),
    ('They ate water yesterday.', 'YESTERDAY THEY WATER EAT FINISH'),
    ("He doesn't cook money.", 'HE MONEY COOK NOT'),
    ('He is reading a mango.', 'HE MANGO READ'),
    ('I am liking a computer.', 'I COMPUTER LIKE'),
    ('They bought water this morning.', 'MORNING THEY WATER BUY FINISH'),
    ("I don't buy a house.", 'I HOUSE BUY NOT'),
    ('The doctor is drinking clothes.', 'DOCTOR CLOTHES DRINK'),
    ('My friend is reading an apple.', 'MY FRIEND APPLE READ'),
    ('We are cooking a house.', 'WE HOUSE COOK'),
    ('He is liking milk.', 'HE MILK LIKE'),
    ("He doesn't eat a computer.", 'HE COMPUTER EAT NOT'),
    ('Why did my father take food?', 'MY FATHER FOOD TAKE WHY'),
    ("My father doesn't buy milk.", 'MY FATHER MILK BUY NOT'),
    ('The teacher is drinking clothes.', 'TEACHER CLOTHES DRINK'),
    ('My friend is taking school.', 'MY FRIEND SCHOOL TAKE'),
    ('The boy is driving a book.', 'BOY BOOK DRIVE'),
    ('My friend is writing an apple.', 'MY FRIEND APPLE WRITE'),
    ("The teacher doesn't read a phone.", 'TEACHER PHONE READ NOT'),
    ("The boy doesn't read a letter.", 'BOY LETTER READ NOT'),
    ('They is drinking food today.', 'TODAY THEY FOOD DRINK'),
    ('I am writing a phone.', 'I PHONE WRITE'),
    ('He is driving a mango.', 'HE MANGO DRIVE'),
    ("My friend doesn't read a bird.", 'MY FRIEND BIRD READ NOT'),
    ('The boy is washing an apple.', 'BOY APPLE WASH'),
    ('He is drinking food.', 'HE FOOD DRINK'),
    ('We are cooking a bird.', 'WE BIRD COOK'),
    ("They don't buy a bird.", 'THEY BIRD BUY NOT'),
    ('Why did my mother take a phone?', 'MY MOTHER PHONE TAKE WHY'),
    ('We are drinking milk.', 'WE MILK DRINK'),
    ('The doctor is giving a computer.', 'DOCTOR COMPUTER GIVE'),
    ('I cooked food yesterday.', 'YESTERDAY I FOOD COOK FINISH'),
    ('My friend is liking a book.', 'MY FRIEND BOOK LIKE'),
    ('I am drinking school.', 'I SCHOOL DRINK'),
    ('We are buying a bird.', 'WE BIRD BUY'),
    ('Where is the doctor going with a book?', 'DOCTOR BOOK GO WHERE'),
    ('I am liking a phone.', 'I PHONE LIKE'),
    ('Where is the girl going with a letter?', 'GIRL LETTER GO WHERE'),
    ('He will cook water next week.', 'NEXT WEEK HE WATER COOK WILL'),
    ('My father is taking a mango.', 'MY FATHER MANGO TAKE'),
    ("We don't buy a bird.", 'WE BIRD BUY NOT'),
    ('The boy is cooking milk.', 'BOY MILK COOK'),
    ('My father is cleaning a phone.', 'MY FATHER PHONE CLEAN'),
    ('The boy is reading a mango.', 'BOY MANGO READ'),
    ('I am reading a book.', 'I BOOK READ'),
    ('My friend is writing milk.', 'MY FRIEND MILK WRITE'),
    ('They are taking an apple.', 'THEY APPLE TAKE'),
    ("I don't eat food.", 'I FOOD EAT NOT'),
    ('They are making clothes.', 'THEY CLOTHES MAKE'),
    ('She is making an apple.', 'SHE APPLE MAKE'),
    ('My mother is eating a letter.', 'MY MOTHER LETTER EAT'),
    ('My father is drinking a computer.', 'MY FATHER COMPUTER DRINK'),
    ('The girl is buying a phone.', 'GIRL PHONE BUY'),
    ('They is reading food tonight.', 'NIGHT THEY FOOD READ'),
    ('He is drinking money.', 'HE MONEY DRINK'),
    ('I am drinking an apple.', 'I APPLE DRINK'),
    ('I am washing school.', 'I SCHOOL WASH'),
    ('My father is cooking money.', 'MY FATHER MONEY COOK'),
    ('I ate a book yesterday.', 'YESTERDAY I BOOK EAT FINISH'),
    ('My mother is driving money.', 'MY MOTHER MONEY DRIVE'),
    ('The doctor is wanting clothes.', 'DOCTOR CLOTHES WANT'),
    ("The teacher doesn't buy a letter.", 'TEACHER LETTER BUY NOT'),
    ('She will drink water next week.', 'NEXT WEEK SHE WATER DRINK WILL'),
    ('Why did my friend take a computer?', 'MY FRIEND COMPUTER TAKE WHY'),
    ('We will buy an apple tomorrow.', 'TOMORROW WE APPLE BUY WILL'),
    ('We are reading a phone.', 'WE PHONE READ'),
    ("My friend doesn't read school.", 'MY FRIEND SCHOOL READ NOT'),
    ('The doctor is giving a mango.', 'DOCTOR MANGO GIVE'),
    ('We is cooking food today.', 'TODAY WE FOOD COOK'),
    ('My father is giving clothes.', 'MY FATHER CLOTHES GIVE'),
    ('We are taking milk.', 'WE MILK TAKE'),
    ('The girl is writing a computer.', 'GIRL COMPUTER WRITE'),
    ('Where is we going with an apple?', 'WE APPLE GO WHERE'),
    ('The doctor is seeing an apple.', 'DOCTOR APPLE SEE'),
    ('My mother is writing money.', 'MY MOTHER MONEY WRITE'),
    ('I am taking school.', 'I SCHOOL TAKE'),
    ('The girl is cooking a phone.', 'GIRL PHONE COOK'),
    ("The teacher doesn't cook water.", 'TEACHER WATER COOK NOT'),
    ('The doctor is making a bird.', 'DOCTOR BIRD MAKE'),
    ('The girl is wanting a mango.', 'GIRL MANGO WANT'),
    ('He is eating food.', 'HE FOOD EAT'),
    ('The boy is eating an apple.', 'BOY APPLE EAT'),
    ('She is drinking a book now.', 'NOW SHE BOOK DRINK'),
    ('My friend is cooking an apple.', 'MY FRIEND APPLE COOK'),
    ('I am reading an apple.', 'I APPLE READ'),
    ('The doctor is washing a book.', 'DOCTOR BOOK WASH'),
    ('My friend is washing a book.', 'MY FRIEND BOOK WASH'),
    ('The doctor is washing milk.', 'DOCTOR MILK WASH'),
    ('The boy is washing a phone.', 'BOY PHONE WASH'),
    ("The boy doesn't read an apple.", 'BOY APPLE READ NOT'),
    ('The doctor is seeing a mango.', 'DOCTOR MANGO SEE'),
    ('We cooked a book this morning.', 'MORNING WE BOOK COOK FINISH'),
    ("The girl doesn't drink clothes.", 'GIRL CLOTHES DRINK NOT'),
    ("My friend doesn't read water.", 'MY FRIEND WATER READ NOT'),
    ('I am wanting a letter.', 'I LETTER WANT'),
    ('My father is drinking food.', 'MY FATHER FOOD DRINK'),
    ('We drank a book yesterday.', 'YESTERDAY WE BOOK DRINK FINISH'),
    ('I will read a book next week.', 'NEXT WEEK I BOOK READ WILL'),
    ('The doctor is seeing a house.', 'DOCTOR HOUSE SEE'),
    ("He doesn't eat a house.", 'HE HOUSE EAT NOT'),
    ('The girl is cooking food.', 'GIRL FOOD COOK'),
    ('I is drinking a book now.', 'NOW I BOOK DRINK'),
    ('We are reading a book.', 'WE BOOK READ'),
]

In [6]:
class Vocab:
    def __init__(self, name):
        self.name = name
        self.word2index = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.index2word = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.word2count = {}
        self.n_words = 4

    def add_sentence(self, sentence):
        for word in sentence.lower().replace(".", "").replace(",", "").split():
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

    def numericalize(self, sentence):
        return [self.word2index.get(word, self.word2index["<unk>"])
                for word in sentence.lower().replace(".", "").replace(",", "").split()]

# Build Vocabularies
src_vocab = Vocab("spoken_text")
trg_vocab = Vocab("isl_gloss")

for src, trg in raw_dataset:
    src_vocab.add_sentence(src)
    trg_vocab.add_sentence(trg)

In [7]:
# Train / Val / Test Split (80/10/10)
train_data, temp_test_data = train_test_split(raw_dataset, test_size=0.2, random_state=SEED)
val_data, test_data = train_test_split(temp_test_data, test_size=0.5, random_state=SEED)

In [8]:
class ISLDataset(Dataset):
    def __init__(self, data, src_vocab, trg_vocab, max_len=15):
        self.data = data
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_sent, trg_sent = self.data[idx]

        src_ids = [self.src_vocab.word2index["<sos>"]] + self.src_vocab.numericalize(src_sent) + [self.src_vocab.word2index["<eos>"]]
        trg_ids = [self.trg_vocab.word2index["<sos>"]] + self.trg_vocab.numericalize(trg_sent) + [self.trg_vocab.word2index["<eos>"]]

        if len(src_ids) < self.max_len:
            src_ids += [self.src_vocab.word2index["<pad>"]] * (self.max_len - len(src_ids))
        else:
            src_ids = src_ids[:self.max_len]

        if len(trg_ids) < self.max_len:
            trg_ids += [self.trg_vocab.word2index["<pad>"]] * (self.max_len - len(trg_ids))
        else:
            trg_ids = trg_ids[:self.max_len]

        return torch.tensor(src_ids), torch.tensor(trg_ids)

In [9]:
MAX_LEN = 12
BATCH_SIZE = 32

train_dataset = ISLDataset(train_data, src_vocab, trg_vocab, MAX_LEN)
val_dataset = ISLDataset(val_data, src_vocab, trg_vocab, MAX_LEN)
test_dataset = ISLDataset(test_data, src_vocab, trg_vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# **Attention-based Seq2Seq Model Architecture**
---

In [10]:
# Attention-based Seq2Seq Model

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, enc_hid_dim, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(enc_hid_dim * 2, dec_hid_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        # Concatenate forward and backward final hidden states to seed the decoder
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        cell = torch.tanh(self.fc(torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1)))
        return outputs, (hidden, cell)

class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return torch.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM((enc_hid_dim * 2) + emb_dim, dec_hid_dim, batch_first=True)
        self.fc_out = nn.Linear((enc_hid_dim * 2) + dec_hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(1)
        embedded = self.dropout(self.embedding(input))
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        weighted = torch.bmm(a, encoder_outputs)
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden.unsqueeze(0), cell.unsqueeze(0)))

        embedded = embedded.squeeze(1)
        output = output.squeeze(1)
        weighted = weighted.squeeze(1)

        prediction = self.fc_out(torch.cat((output, weighted, embedded), dim=1))
        return prediction, hidden.squeeze(0), cell.squeeze(0)

In [11]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        encoder_outputs, (hidden, cell) = self.encoder(src)

        input = trg[:, 0]
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs

In [12]:
# Model Initialization
INPUT_DIM = src_vocab.n_words
OUTPUT_DIM = trg_vocab.n_words
ENC_EMB_DIM = 64
DEC_EMB_DIM = 64
ENC_HID_DIM = 64
DEC_HID_DIM = 64
ENC_DROPOUT = 0.3
DEC_DROPOUT = 0.3

attn = Attention(ENC_HID_DIM, DEC_HID_DIM)
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, DEC_DROPOUT, attn)
model_lstm = Seq2Seq(enc, dec, device).to(device)

In [13]:
#Training & Best Model Checkpointing
optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=trg_vocab.word2index["<pad>"])

N_EPOCHS = 50
best_valid_loss = float('inf')
checkpoint_path = "best_model_seq2seq.pt"

for epoch in range(N_EPOCHS):
    model_lstm.train()
    epoch_loss = 0
    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model_lstm(src, trg)

        output_dim = output.shape[-1]
        output = output[:, 1:, :].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    model_lstm.eval()
    valid_loss = 0
    with torch.no_grad():
        for src, trg in val_loader:
            src, trg = src.to(device), trg.to(device)
            output = model_lstm(src, trg, teacher_forcing_ratio=0.0) # No teacher forcing during validation
            output_dim = output.shape[-1]
            output = output[:, 1:, :].reshape(-1, output_dim)
            trg = trg[:, 1:].reshape(-1)
            loss = criterion(output, trg)
            valid_loss += loss.item()

    train_loss_avg = epoch_loss / len(train_loader)
    valid_loss_avg = valid_loss / len(val_loader)

    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss_avg:.3f} | Val Loss: {valid_loss_avg:.3f}')

    if valid_loss_avg < best_valid_loss:
        best_valid_loss = valid_loss_avg
        torch.save(model_lstm.state_dict(), checkpoint_path)
        print(f" -> Saved new best model checkpoint to {checkpoint_path}")

Epoch: 01 | Train Loss: 3.851 | Val Loss: 3.147
 -> Saved new best model checkpoint to best_model_seq2seq.pt
Epoch: 02 | Train Loss: 2.895 | Val Loss: 2.834
 -> Saved new best model checkpoint to best_model_seq2seq.pt
Epoch: 03 | Train Loss: 2.558 | Val Loss: 2.480
 -> Saved new best model checkpoint to best_model_seq2seq.pt
Epoch: 04 | Train Loss: 2.162 | Val Loss: 1.978
 -> Saved new best model checkpoint to best_model_seq2seq.pt
Epoch: 05 | Train Loss: 1.766 | Val Loss: 1.591
 -> Saved new best model checkpoint to best_model_seq2seq.pt
Epoch: 06 | Train Loss: 1.458 | Val Loss: 1.328
 -> Saved new best model checkpoint to best_model_seq2seq.pt
Epoch: 07 | Train Loss: 1.232 | Val Loss: 1.100
 -> Saved new best model checkpoint to best_model_seq2seq.pt
Epoch: 08 | Train Loss: 1.019 | Val Loss: 0.936
 -> Saved new best model checkpoint to best_model_seq2seq.pt
Epoch: 09 | Train Loss: 0.862 | Val Loss: 0.819
 -> Saved new best model checkpoint to best_model_seq2seq.pt
Epoch: 10 | Train L

In [14]:


# Load best weights
model_lstm.load_state_dict(torch.load(checkpoint_path))
model_lstm.eval()

references = []
hypotheses = []

with torch.no_grad():
    for src, trg in test_loader:
        src = src.to(device)
        encoder_outputs, (hidden, cell) = model_lstm.encoder(src)
        batch_size = src.shape[0]

        input_tokens = torch.full((batch_size,), trg_vocab.word2index["<sos>"], dtype=torch.long, device=device)
        generated_outputs = []

        for t in range(1, MAX_LEN):
            output, hidden, cell = model_lstm.decoder(input_tokens, hidden, cell, encoder_outputs)
            top1 = output.argmax(1)
            generated_outputs.append(top1.cpu().numpy())
            input_tokens = top1

        generated_outputs = np.array(generated_outputs).T

        for i in range(batch_size):
            ref_tokens = [trg_vocab.index2word[idx.item()] for idx in trg[i]
                          if idx.item() not in [trg_vocab.word2index["<pad>"], trg_vocab.word2index["<sos>"], trg_vocab.word2index["<eos>"]]]

            hyp_tokens = []
            for idx in generated_outputs[i]:
                word = trg_vocab.index2word[idx]
                if word == "<eos>":
                    break
                if word not in ["<pad>", "<sos>"]:
                    hyp_tokens.append(word)

            references.append([ref_tokens])
            hypotheses.append(hyp_tokens)

smoothie = SmoothingFunction().method1
bleu1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0), smoothing_function=smoothie) * 100
bleu2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie) * 100
bleu3 = corpus_bleu(references, hypotheses, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothie) * 100
bleu4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie) * 100

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = []

for ref, hyp in zip(references, hypotheses):
    ref_str = " ".join(ref[0])
    hyp_str = " ".join(hyp)
    score = scorer.score(ref_str, hyp_str)
    rouge_scores.append(score['rougeL'].fmeasure)

mean_rouge_l = np.mean(rouge_scores) * 100

print("\n=== FINAL TEST EVALUATION METRICS ===")
print(f"BLEU-1: {bleu1:.2f}%")
print(f"BLEU-2: {bleu2:.2f}%")
print(f"BLEU-3: {bleu3:.2f}%")
print(f"BLEU-4: {bleu4:.2f}%")
print(f"ROUGE-L: {mean_rouge_l:.2f}%")


=== FINAL TEST EVALUATION METRICS ===
BLEU-1: 98.43%
BLEU-2: 97.68%
BLEU-3: 97.01%
BLEU-4: 86.58%
ROUGE-L: 98.53%


# **Vanilla Transformer**

In [15]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50, dropout=0.25):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class VanillaTransformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, d_model=128, nhead=4,
                 num_encoder_layers=3, num_decoder_layers=3, dim_feedforward=256, dropout=0.25):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=50, dropout=dropout)
        self.pos_decoder = PositionalEncoding(d_model, max_len=50, dropout=dropout)

        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers, num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True
        )
        self.fc_out = nn.Linear(d_model, trg_vocab_size)

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, src, trg):
        src_padding_mask = (src == 0)
        trg_padding_mask = (trg == 0)
        trg_mask = self.generate_square_subsequent_mask(trg.size(1)).to(src.device)

        src_emb = self.pos_encoder(self.src_embedding(src) * np.sqrt(self.d_model))
        trg_emb = self.pos_decoder(self.trg_embedding(trg) * np.sqrt(self.d_model))

        out = self.transformer(
            src_emb, trg_emb,
            tgt_mask=trg_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=trg_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )
        return self.fc_out(out)

In [16]:
class NoamScheduler:
    def __init__(self, optimizer, d_model, warmup_steps=400):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.step_num = 0

    def step(self):
        self.step_num += 1
        lr = (self.d_model ** -0.5) * min(self.step_num ** -0.5, self.step_num * (self.warmup_steps ** -1.5))
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

In [17]:
model_tfm = VanillaTransformer(src_vocab.n_words, trg_vocab.n_words).to(device)
optimizer = optim.Adam(model_tfm.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
scheduler = NoamScheduler(optimizer, d_model=128, warmup_steps=300)
criterion = nn.CrossEntropyLoss(ignore_index=0)

N_EPOCHS = 50
best_valid_loss = float('inf')
checkpoint_path = "best_transformer_model.pt"

for epoch in range(N_EPOCHS):
    model_tfm.train()
    epoch_loss = 0
    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)

        trg_input = trg[:, :-1]
        trg_output = trg[:, 1:]

        optimizer.zero_grad()
        output = model_tfm(src, trg_input)

        loss = criterion(output.reshape(-1, output.shape[-1]), trg_output.reshape(-1))
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model_tfm.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()

    # Evaluation
    model_tfm.eval()
    valid_loss = 0
    with torch.no_grad():
        for src, trg in val_loader:
            src, trg = src.to(device), trg.to(device)
            trg_input = trg[:, :-1]
            trg_output = trg[:, 1:]
            output = model_tfm(src, trg_input)
            loss = criterion(output.reshape(-1, output.shape[-1]), trg_output.reshape(-1))
            valid_loss += loss.item()

    train_loss_avg = epoch_loss / len(train_loader)
    valid_loss_avg = valid_loss / len(val_loader)
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss_avg:.4f} | Val Loss: {valid_loss_avg:.4f}")

    if valid_loss_avg < best_valid_loss:
        best_valid_loss = valid_loss_avg
        torch.save(model_tfm.state_dict(), checkpoint_path)
        print(f" -> Checkpoint saved to {checkpoint_path}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 01 | Train Loss: 4.2972 | Val Loss: 3.7123
 -> Checkpoint saved to best_transformer_model.pt
Epoch 02 | Train Loss: 3.2935 | Val Loss: 2.5689
 -> Checkpoint saved to best_transformer_model.pt
Epoch 03 | Train Loss: 2.4591 | Val Loss: 1.7310
 -> Checkpoint saved to best_transformer_model.pt
Epoch 04 | Train Loss: 1.8641 | Val Loss: 1.3576
 -> Checkpoint saved to best_transformer_model.pt
Epoch 05 | Train Loss: 1.5354 | Val Loss: 1.1915
 -> Checkpoint saved to best_transformer_model.pt
Epoch 06 | Train Loss: 1.3881 | Val Loss: 1.0210
 -> Checkpoint saved to best_transformer_model.pt
Epoch 07 | Train Loss: 1.4381 | Val Loss: 1.1008
Epoch 08 | Train Loss: 1.6146 | Val Loss: 1.3353
Epoch 09 | Train Loss: 1.6970 | Val Loss: 1.2236
Epoch 10 | Train Loss: 1.8394 | Val Loss: 1.3112
Epoch 11 | Train Loss: 1.9210 | Val Loss: 1.5900
Epoch 12 | Train Loss: 1.8920 | Val Loss: 1.4550
Epoch 13 | Train Loss: 1.8636 | Val Loss: 1.4265
Epoch 14 | Train Loss: 1.8005 | Val Loss: 1.4590
Epoch 15 | Tra

In [18]:
model_tfm.load_state_dict(torch.load(checkpoint_path))
model_tfm.eval()

references, hypotheses = [], []

with torch.no_grad():
    for src, trg in test_loader:
        src = src.to(device)
        batch_size = src.shape[0]

        ys = torch.full((batch_size, 1), 1, dtype=torch.long, device=device)

        for _ in range(MAX_LEN - 1):
            out = model_tfm(src, ys)
            next_word = out[:, -1, :].argmax(dim=-1, keepdim=True)
            ys = torch.cat([ys, next_word], dim=1)

        ys = ys.cpu().numpy()
        trg = trg.cpu().numpy()

        for b in range(batch_size):
            ref_tokens = [trg_vocab.index2word[idx] for idx in trg[b] if idx not in [0, 1, 2]]
            hyp_tokens = []
            for idx in ys[b][1:]:
                if idx == 2:
                  break
                if idx not in [0, 1]:
                    hyp_tokens.append(trg_vocab.index2word[idx])

            references.append([ref_tokens])
            hypotheses.append(hyp_tokens)

# Final Metric Estimations
smoothie = SmoothingFunction().method1
b1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0), smoothing_function=smoothie) * 100
b2 = corpus_bleu(references, hypotheses, weights=(0.5, 0.5, 0, 0), smoothing_function=smoothie) * 100
b3 = corpus_bleu(references, hypotheses, weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothie) * 100
b4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie) * 100

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = [scorer.score(" ".join(ref[0]), " ".join(hyp))['rougeL'].fmeasure for ref, hyp in zip(references, hypotheses)]

print("\n=== TRANSFORMER TEST METRICS ===")
print(f"BLEU-1: {b1:.2f}% | BLEU-2: {b2:.2f}% | BLEU-3: {b3:.2f}% | BLEU-4: {b4:.2f}%")
print(f"ROUGE-L: {np.mean(rouge_scores) * 100:.2f}%")


=== TRANSFORMER TEST METRICS ===
BLEU-1: 63.85% | BLEU-2: 51.26% | BLEU-3: 40.04% | BLEU-4: 23.33%
ROUGE-L: 66.47%


In [19]:
# Run Autoregressive Inference & Compute Latency
import time
def evaluate_model_performance(model, model_type, test_loader, trg_vocab, max_len=12):
    model.eval()
    references = []
    hypotheses = []
    latencies = []

    with torch.no_grad():
        for src, trg in test_loader:
            src = src.to(device)
            batch_size = src.shape[0]

            start_time = time.perf_counter()

            if model_type == "lstm":
                encoder_outputs, (hidden, cell) = model.encoder(src)
                input_tokens = torch.full((batch_size,), trg_vocab.word2index["<sos>"], dtype=torch.long, device=device)
                generated_outputs = []
                for t in range(1, max_len):
                    output, hidden, cell = model.decoder(input_tokens, hidden, cell, encoder_outputs)
                    top1 = output.argmax(1)
                    generated_outputs.append(top1.cpu().numpy())
                    input_tokens = top1
                generated_outputs = np.array(generated_outputs).T

            elif model_type == "transformer":
                ys = torch.full((batch_size, 1), 1, dtype=torch.long, device=device)
                for _ in range(max_len - 1):
                    out = model(src, ys)
                    next_word = out[:, -1, :].argmax(dim=-1, keepdim=True)
                    ys = torch.cat([ys, next_word], dim=1)
                generated_outputs = ys[:, 1:].cpu().numpy()

            end_time = time.perf_counter()
            latencies.append(((end_time - start_time) / batch_size) * 1000)

            trg = trg.cpu().numpy()
            for b in range(batch_size):
                ref_tokens = [trg_vocab.index2word[idx] for idx in trg[b] if idx not in [0, 1, 2]]
                hyp_tokens = []
                for idx in generated_outputs[b]:
                    if idx == 2:
                      break
                    if idx not in [0, 1]:
                        hyp_tokens.append(trg_vocab.index2word[idx])
                references.append([ref_tokens])
                hypotheses.append(hyp_tokens)

    # Compute Final Scores
    smoothie = SmoothingFunction().method1
    b1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0), smoothing_function=smoothie) * 100
    b4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie) * 100

    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_scores = [scorer.score(" ".join(ref[0]), " ".join(hyp))['rougeL'].fmeasure for ref, hyp in zip(references, hypotheses)]

    return b1, b4, np.mean(rouge_scores) * 100, np.mean(latencies)

lstm_model_path = "best_model_seq2seq.pt"
trans_model_path = "best_transformer_model.pt"

model_lstm.load_state_dict(torch.load(lstm_model_path, map_location=device))
model_tfm.load_state_dict(torch.load(trans_model_path, map_location=device))

l_b1, l_b4, l_rouge, l_lat = evaluate_model_performance(model_lstm, "lstm", test_loader, trg_vocab, MAX_LEN)
t_b1, t_b4, t_rouge, t_lat = evaluate_model_performance(model_tfm, "transformer", test_loader, trg_vocab, MAX_LEN)
print(f"LSTM: BLEU-1: {l_b1:.2f}% | BLEU-4: {l_b4:.2f}% | ROUGE-L: {l_rouge:.2f}% | Latency: {l_lat:.2f} ms")
print(f"TRANSFORMER: BLEU-1: {t_b1:.2f}% | BLEU-4: {t_b4:.2f}% | ROUGE-L: {t_rouge:.2f}% | Latency: {t_lat:.2f} ms")

LSTM: BLEU-1: 98.43% | BLEU-4: 86.58% | ROUGE-L: 98.53% | Latency: 0.31 ms
TRANSFORMER: BLEU-1: 63.85% | BLEU-4: 23.33% | ROUGE-L: 66.47% | Latency: 5.49 ms


In [20]:
def analyze_model_failures(lstm_model, transformer_model, test_loader, src_vocab, trg_vocab, max_len=12):
    lstm_model.eval()
    transformer_model.eval()
    smoothie = SmoothingFunction().method1

    analysis_records = []
    with torch.no_grad():
        for src, trg in test_loader:
            src_device = src.to(device)
            batch_size = src.shape[0]

            encoder_outputs, (hidden, cell) = lstm_model.encoder(src_device)
            input_tokens = torch.full((batch_size,), trg_vocab.word2index["<sos>"], dtype=torch.long, device=device)
            lstm_outputs = []
            for t in range(1, max_len):
                output, hidden, cell = lstm_model.decoder(input_tokens, hidden, cell, encoder_outputs)
                top1 = output.argmax(1)
                lstm_outputs.append(top1.cpu().numpy())
                input_tokens = top1
            lstm_outputs = np.array(lstm_outputs).T

            ys = torch.full((batch_size, 1), 1, dtype=torch.long, device=device)
            for _ in range(max_len - 1):
                out = transformer_model(src_device, ys)
                next_word = out[:, -1, :].argmax(dim=-1, keepdim=True)
                ys = torch.cat([ys, next_word], dim=1)
            trans_outputs = ys[:, 1:].cpu().numpy()

            src = src.numpy()
            trg = trg.numpy()

            for b in range(batch_size):
                src_words = [src_vocab.index2word[idx] for idx in src[b] if idx not in [0, 1, 2]]
                src_sent = " ".join(src_words)

                ref_words = [trg_vocab.index2word[idx] for idx in trg[b] if idx not in [0, 1, 2]]

                lstm_words = []
                for idx in lstm_outputs[b]:
                    if idx == 2: break
                    if idx not in [0, 1]: lstm_words.append(trg_vocab.index2word[idx])

                trans_words = []
                for idx in trans_outputs[b]:
                    if idx == 2: break
                    if idx not in [0, 1]: trans_words.append(trg_vocab.index2word[idx])

                lstm_bleu = sentence_bleu([ref_words], lstm_words, weights=(1, 0, 0, 0), smoothing_function=smoothie) * 100
                trans_bleu = sentence_bleu([ref_words], trans_words, weights=(1, 0, 0, 0), smoothing_function=smoothie) * 100

                analysis_records.append({
                    'source': src_sent,
                    'reference': " ".join(ref_words),
                    'lstm_pred': " ".join(lstm_words),
                    'trans_pred': " ".join(trans_words),
                    'lstm_bleu': lstm_bleu,
                    'trans_bleu': trans_bleu,
                    'delta': trans_bleu - lstm_bleu
                })

    return analysis_records

records = analyze_model_failures(model_lstm, model_tfm, test_loader, src_vocab, trg_vocab, MAX_LEN)

In [21]:
print(f"Records: {records}")

Records: [{'source': 'we is cooking a book tonight', 'reference': 'night we book cook', 'lstm_pred': 'night we book cook', 'trans_pred': 'we book cook', 'lstm_bleu': 100.0, 'trans_bleu': 71.65313105737893, 'delta': -28.34686894262107}, {'source': 'the doctor is drinking clothes', 'reference': 'doctor clothes drink', 'lstm_pred': 'doctor clothes drink', 'trans_pred': 'doctor apple drink', 'lstm_bleu': 100.0, 'trans_bleu': 66.66666666666666, 'delta': -33.33333333333334}, {'source': 'why did i take a computer?', 'reference': 'i computer take why', 'lstm_pred': 'i computer take why', 'trans_pred': 'i letter take', 'lstm_bleu': 100.0, 'trans_bleu': 47.76875403825262, 'delta': -52.23124596174738}, {'source': 'i will eat a book tomorrow', 'reference': 'tomorrow i book eat will', 'lstm_pred': 'tomorrow i book eat will', 'trans_pred': 'i book read', 'lstm_bleu': 100.0, 'trans_bleu': 34.227807935506135, 'delta': -65.77219206449386}, {'source': 'we bought an apple this morning', 'reference': 'mor

In [22]:
!git config --global user.email "abhaykumar280799@gmail.com"
!git config --global user.name  "Abhay-287"

In [23]:
!git clone https://github.com/Abhay-287/Text-to-ISL-comparative-study

Cloning into 'Text-to-ISL-comparative-study'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), done.


In [24]:
!cp /content/*.ipynb Text-to-ISL-comparative-study/

cp: cannot stat '/content/*.ipynb': No such file or directory
